<a href="https://colab.research.google.com/github/wangwangwang77/Machine-learning-in-UM/blob/main/main_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade --no-cache-dir tensorflow keras

In [ ]:
import tensorflow as tf
print("TensorFlow Version:", tf.__version__)

import numpy as np
print("NumPy Version:", np.__version__)
print("Has 'dtypes':", hasattr(np, "dtypes"))  # This should return True


TensorFlow Version: 2.21.0
NumPy Version: 2.0.2
Has 'dtypes': True


In [ ]:
import sys
import pandas as pd
import os
from functools import reduce
import keras
from tensorflow.keras.layers import Normalization
from google.colab import drive

# Mount your Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install openpyxl
!pip install tabulate
from tabulate import tabulate

In [ ]:
from scipy.stats import pearsonr
#import statsmodels.api as sm
#from statsmodels.tsa.api import VAR
import matplotlib.pyplot as plt
#import statsmodels.api as sm

import numpy as np
from typing import Dict, Any, List, Optional
#from statsmodels.tsa.stattools import acf

class economy:

    """
    Parameters:

    """
    def __init__(self, numAssets, numStates, numPeriods_employ, numPeriods_retire, numSim):

        # pass information for the simulation
        self.numPeriods_employ = numPeriods_employ
        self.numPeriods_retire = numPeriods_retire
        self.numPeriods = self.numPeriods_employ + self.numPeriods_retire
        self.numAssets = numAssets
        self.numStates = numStates
        self.numSim = numSim

    def simulation(self, parModel):

        simData = eval("self."+parModel["model"])(parModel)

        return simData

    def simulate_primal_market_array(self,parModel):

        K_returns = 101         # number of returns per path (k = 0,...,50)
        K_prices  = K_returns + 1   # 52 prices per path (S_0,...,S_51)

        # -----def simulate_primal_market_array(self, parModel):

        # ----- setup -----
        N = 1                 # number of risky assets
        n_paths = 1024
        T = 10.0
        B0 = 1.0

        dt = T / 100.0          # time step is T/50 as you require
        #toy term structures per return-period k = 0,...,50 -----
        # r_t = 0.01, sigma_t = I_N, lambda_t = 0.2
        r_t = np.zeros(K_prices, dtype=float) + 0.01          # (51,)

        sigma_base = np.eye(N, dtype=float) * 0.25                   # (N, N)
        sigma_t = np.repeat(sigma_base[None, :, :],
                            K_prices, axis=0)                 # (51, N, N)

        lambda_t = np.zeros((K_prices, N), dtype=float) + 0.2 # (51, N)

        # initial risky prices
        S0 = np.ones(N, dtype=float)

        # RNG
        seed = parModel.get("seed", 1234)
        rng = np.random.default_rng(seed)

        # sanity check
        n_assets = S0.shape[0]
        assert sigma_t.shape == (K_prices, n_assets, n_assets)
        assert lambda_t.shape == (K_prices, n_assets)

        # ----- mu_t from r_t, sigma_t, lambda_t -----
        mu_t = np.empty((K_prices, n_assets), dtype=float)
        for k in range(K_prices):
            mu_t[k] = r_t[k] + sigma_t[k] @ lambda_t[k]   # (n_assets,)

        # ----- risk-free simple returns for each period k=0..50 -----
        R_rf = np.exp(r_t[:-1] * dt)                    # (51,)

        # ----- time feature: 51 meaningful times 0,...,1 -----
        t_raw = np.arange(K_returns, dtype="float32") / 100.0   # 0, 1/50, ..., 50/50

        # ----- simulate paths and collect rows -----
        all_rows = []

        for path in range(n_paths*parModel["numBatches"]):
            # prices S_0,...,S_51  (52 prices)
            S = np.empty((K_prices, n_assets), dtype=float)
            B = np.empty(K_prices, dtype=float)
            S[0] = S0
            B[0] = B0

            risky_returns = np.empty(K_returns, dtype=float)   # 51 returns

            for k in range(K_returns):   # k = 0,...,50
                # Brownian increment ~ N(0, dt I_N)
                dW = rng.normal(loc=0.0, scale=np.sqrt(dt), size=n_assets)  # (N,)

                sigma_k = sigma_t[k]   # (N, N)
                mu_k    = mu_t[k]      # (N,)

                diffusion   = sigma_k @ dW                    # (N,)
                sigma_norm2 = np.sum(sigma_k**2, axis=1)      # (N,)
                drift       = (mu_k - 0.5 * sigma_norm2) * dt # (N,)

                log_increment = drift + diffusion
                S[k+1] = S[k] * np.exp(log_increment)

                # simple risky return for asset 0 over this period
                risky_returns[k] = S[k+1, 0] / S[k, 0]

                # bank account over this period
                B[k+1] = B[k] * np.exp(r_t[k] * dt)

            # per-path block: shape (51, 3)
            path_block = np.column_stack([
                risky_returns,   # (51,)
                R_rf,            # (51,)
                t_raw,          # (51,)
            ])
            all_rows.append(path_block)

        # stack over paths → ((51 * n_paths), 3)
        data = np.vstack(all_rows)

        return data

    def simulate_market_ou_array(self, parModel: Dict[str, Any]):
        N = self.numAssets
        K = self.numPeriods
        n_paths = self.numSim * parModel.get("numBatches", 1)

        T = 2.0
        dt = T / K
        sqrt_dt = np.sqrt(dt)

        r0 = float(parModel.get("r0", 0.02))
        kappa_r = float(parModel.get("kappa_r", 1.25))
        r_bar = float(parModel.get("r_bar", 0.03))
        sigma_r = float(parModel.get("sigma_r", 0.02))

        mu = np.asarray(parModel["mu"], dtype=float).reshape(N)
        sigma_diag = np.asarray(parModel["sigma_diag"], dtype=float).reshape(N)

        if np.any(sigma_diag <= 0.0):
            raise ValueError("All diagonal entries of sigma must be strictly positive.")

        seed = int(parModel.get("seed", 1234))
        exact_short_rate = bool(parModel.get("exact_short_rate", True))
        return_type = parModel.get("return_type", "gross")      # "gross" or "simple"
        time_type = parModel.get("time_type", "normalized")         # "actual" or "normalized"

        rng = np.random.default_rng(seed)

        # -------------------------
        # 2. Time column
        # -------------------------
        if time_type == "actual":
            time_col_single = np.arange(K, dtype=float) * dt
        elif time_type == "normalized":
            time_col_single = np.arange(K, dtype=float) / K
        else:
            raise ValueError("time_type must be either 'actual' or 'normalized'.")

        # -------------------------
        # 3. Exact OU discretization constants
        # -------------------------
        use_exact_ou = exact_short_rate and (abs(kappa_r) > 1e-12)
        if use_exact_ou:
            exp_kdt = np.exp(-kappa_r * dt)
            ou_std = sigma_r * np.sqrt((1.0 - np.exp(-2.0 * kappa_r * dt)) / (2.0 * kappa_r))

        # -------------------------
        # 4. Simulate and stack rows
        # -------------------------
        all_rows = []

        for path in range(n_paths):
            r_t = r0

            for k in range(K):  #K=0,1,...,50
                # risk-free gross return over [t_k, t_{k+1}]
                R_f = np.exp(r_t * dt)

                # risky gross returns over [t_k, t_{k+1}]
                z = rng.normal(loc=0.0, scale=1.0, size=N)
                dW = sqrt_dt * z
                risky_gross = np.exp((mu - 0.5 * sigma_diag**2) * dt + sigma_diag * dW)

                if return_type == "gross":
                    risky_out = risky_gross
                    rf_out = R_f
                elif return_type == "simple":
                    risky_out = risky_gross - 1.0
                    rf_out = R_f - 1.0
                else:
                    raise ValueError("return_type must be either 'gross' or 'simple'.")

                row = np.concatenate([risky_out, np.array([rf_out, time_col_single[k]])])
                all_rows.append(row)

                # update short rate
                eps_r = rng.normal()
                if use_exact_ou:
                    r_t = r_bar + (r_t - r_bar) * exp_kdt + ou_std * eps_r
                else:
                    r_t = r_t + kappa_r * (r_bar - r_t) * dt + sigma_r * sqrt_dt * eps_r

        data = np.vstack(all_rows)

        return data


    def bootstrap_data_Ic_9_employ(self, parModel):

        numBatches = parModel["numBatches"]
        expected_block_length = 8
        maturity = (self.numPeriods_employ + 1) * self.numSim * numBatches  # New fixed number of samples for each bootstrap draw
        p = 1 / expected_block_length
        rng = np.random.default_rng(7)

        drive.mount("/content/drive", force_remount=True)
        file_path = '/content/drive/My Drive/industry_porfolio(new).xlsx'

        df_IP = pd.read_excel(file_path,sheet_name="indus_return",engine="openpyxl")
        df_IP = df_IP.iloc[1:]
        df_IP.columns = ['Year', 'Construction_Ip', 'Finance_Ip', 'Manuf_Ip','Mining_Ip', 'retail_Ip', 'Service_Ip','Comm. Util._Ip', 'wholesale_Ip']
        df_IP = df_IP.map(lambda x: str(x).replace('\xa0', '') if isinstance(x, str) else x)
        #df_IP = df_IP.applymap(lambda x: str(x).replace('\xa0', '') if isinstance(x, str) else x)
        df_IP= df_IP.apply(pd.to_numeric, errors='coerce')
        df_IP.iloc[:, 1:] = df_IP.iloc[:, 1:]/ 100 + 1
        df_IP.iloc[:, 1:] = df_IP.iloc[:, 1:].round(6)

        df_ME = pd.read_excel(file_path,sheet_name="ME",engine="openpyxl")
        df_ME = df_ME.iloc[1:]
        df_ME.columns = ['Year', 'Construction_ME', 'Finance_ME', 'Manuf_ME','Mining_ME', 'retail_ME', 'Service_ME','Comm. Util._ME', 'wholesale_ME']

        # --- Align frames on Year and make Year the index ---
        ip = df_IP.copy().set_index('Year')
        me = df_ME.copy().set_index('Year')

        # Get the common industry base names by stripping the suffixes
        ip_cols = [c.replace('_Ip','') for c in ip.columns]
        me_cols = [c.replace('_ME','') for c in me.columns]
        assert set(ip_cols) == set(me_cols), "Industry sets differ between returns and ME."

        # Reorder columns consistently by the returns order
        industries = ip_cols
        ip = ip[[f"{ind}_Ip" for ind in industries]]
        me = me[[f"{ind}_ME" for ind in industries]]

        # Convenience: put returns and ME in matching wide DataFrames with same column names
        R = ip.copy()
        R.columns = industries
        W = me.copy()
        W.columns = industries

        # --- Compute R_{M/k} for each exclusion k ---
        df_mk = pd.DataFrame(index=R.index)

        for k in industries:
            # exclude industry k
            R_excl = R.drop(columns=k)
            W_excl = W.drop(columns=k)

            # row-wise normalized weights excluding k
            denom = W_excl.sum(axis=1)
            w_norm = W_excl.div(denom, axis=0)

            # weighted average gross return excluding k
            df_mk[f"R_M_excl_{k}"] = (w_norm * R_excl).sum(axis=1)

        # Put Year back as a column (optional)
        df_mk = df_mk.reset_index()

        file_path = '/content/drive/My Drive/project data/F-F_Research_Data_Factors.xlsx'
        df1 = pd.read_excel(file_path, engine="openpyxl")# Try different delimiters if neede
        df1 = df1.iloc[1:]
        df1.columns = ['Year', 'Mkt', 'RF']
        df1['Mkt']=  df1['Mkt']/100 + 1
        df1['RF'] = df1['RF']/100 + 1

        file_path_cpi = '/content/drive/My Drive/project data/CPI.xlsx'
        df_If = pd.read_excel(file_path_cpi, sheet_name="sheet2", engine="openpyxl")
        df_If.columns = ['Year', 'inflation']

#################################################################  just create the predictors
        df_If_lag = df_If.copy()
        df_If_lag['Year'] =  df_If_lag['Year'] + 1

        new_df1_lag = df1[['Year', 'Mkt']].copy()
        new_df1_lag['Year'] = new_df1_lag['Year'] +  1

        df_IP_lag = df_IP.copy()
        df_IP_lag['Year'] = df_IP_lag['Year'] + 1
        df_forecast = [new_df1_lag,df_IP_lag,df_If_lag]

        df_IP_lag = reduce(lambda left, right: pd.merge(left, right, on='Year', how='inner'),df_forecast)

        df_IP_lag.columns = ['Year','Mkt_lag', 'Construction_Ip_lag', 'Finance_Ip_lag', 'Manuf_Ip_lag','Mining_Ip_lag', 'retail_Ip_lag', 'Service_Ip_lag','Comm. Util._Ip_lag', 'wholesale_Ip_lag','inflation']
        select_columns = ['Mkt_lag', 'Construction_Ip_lag', 'Finance_Ip_lag', 'Manuf_Ip_lag','Mining_Ip_lag', 'retail_Ip_lag', 'Service_Ip_lag','Comm. Util._Ip_lag', 'wholesale_Ip_lag']

        df_IP_lag[select_columns] = df_IP_lag[select_columns].apply(
            lambda x: x.values /  df_IP_lag['inflation'].values)
        df_IP_lag.drop(columns=['inflation'], inplace=True)

######################################################################

        file_path_IC = '/content/drive/My Drive/project data/split_yearlyPerCapitaWages(new).xlsx'

        df_IC = pd.read_excel(file_path_IC, sheet_name="sheet1", engine="openpyxl")
        df_IC.columns = ['Year', 'Construction', 'Finance', 'Manufacturing', 'Mining', 'retail',
                        'Services', 'Comm. Util.', 'Wholesale', 'Government']

        select_columns = ['Construction', 'Finance', 'Manufacturing', 'Mining', 'retail', 'Services',
                          'Comm. Util.', 'Wholesale','Government']

        df_IC[select_columns] = df_IC[select_columns].apply(
            lambda x: x[1:].values / x[:-1].values)

        df_IC_lagged = df_IC.copy()
        df_IC_lagged['Year'] = df_IC_lagged['Year'] + 1

        file_path_dp = '/content/drive/My Drive/project data/predicor_industry.xlsx'

        df_dp = pd.read_excel(file_path_dp, sheet_name="Sheet1", usecols="A:J", engine="openpyxl")
        df_dp.columns = ["Year", "dp_market", "dp_constr","Fin_dp","dp_manuf","Mine_dp","Retail_dp","Service_dp","Comm. Util_dp","Wholesale_dp"]
        # Reset the index, if necessary
        df_dp = df_dp.reset_index(drop=True)
        #df_dp['Year'] = pd.to_numeric(df_dp['Year'], errors='coerce')
        #df_IC.iloc[:, 1:] = df_IC.iloc[:, 1:] + 1

        df_dp_lag = df_dp.copy()
        df_dp_lag['Year'] = df_dp_lag['Year'] + 1
        df_dp_lag.columns = ["Year", "dp_market_lag", "dp_constr_lag","Fin_dp_lag","dp_manuf_lag","Mine_dp_lag","Retail_dp_lag","Service_dp_lag","Comm. Util_dp_lag","Wholesale_dp_lag"]

        #merged_df_fin = pd.merge(df_IP_lead[['Year', 'Finance_Ip_lead']],df_dp[['Year', 'Fin_dp']],on='Year',how='inner')
        #merged_df_mkt = pd.merge(df_IP_lead[['Year', 'Mkt_lead']],df_dp[['Year', 'dp_market']],on='Year',how='inner')
        #plt.scatter(merged_df['Fin_dp'],merged_df['Finance_Ip'], color='blue', marker='o', s=100)

        # Add titles and labels
        #plt.title("Fin_dp vs Finance_Ip Scatter Plot")
        #plt.xlabel("Finance_Ip")
        #plt.ylabel("Fin_dp")
        #plt.show()

        dfs_employ = [df_IP,df_mk,df1, df_dp, df_IC_lagged,df_IP_lag,df_dp_lag,df_If]

        merged_dfs_employ = reduce(lambda left, right: pd.merge(left, right, on='Year', how='inner'),dfs_employ)

        select_columns = ['Construction_Ip', 'Finance_Ip', 'Manuf_Ip','Mining_Ip', 'retail_Ip', 'Service_Ip','Comm. Util._Ip', 'wholesale_Ip','Construction', 'Finance', 'Manufacturing', 'Mining', 'retail', 'Services',
                          'Comm. Util.', 'Wholesale','Government','Mkt', 'RF','R_M_excl_Construction', 'R_M_excl_Finance', 'R_M_excl_Manuf','R_M_excl_Mining', 'R_M_excl_retail', 'R_M_excl_Service','R_M_excl_Comm. Util.', 'R_M_excl_wholesale']
        merged_dfs_employ[select_columns] = merged_dfs_employ[select_columns].apply(
            lambda x: x.values / merged_dfs_employ['inflation'].values)

        mean = merged_dfs_employ['Finance'].mean()
        factor = 2
        #merged_dfs_employ['Finance']  = mean + factor * (merged_dfs_employ['Finance'] - mean)
        merged_dfs_employ.drop(columns=['inflation'], inplace=True)
        merged_dfs_employ = merged_dfs_employ.iloc[:-1]

        ##################################
        summary_stats = merged_dfs_employ.agg(['mean', 'std', 'skew', 'kurtosis', 'min', 'max', 'count'])
        print(tabulate(summary_stats, headers='keys', tablefmt='pretty'))
        ################################

        """
        #print(merged_df)

        # Define two sets of columns
        set1 = ['Construction_Ip', 'Finance_Ip', 'Manuf_Ip','Mining_Ip', 'retail_Ip', 'Service_Ip','Comm. Util._Ip', 'wholesale_Ip','Mkt', 'RF']
        set2 = ['Construction', 'Finance', 'Manufacturing', 'Mining', 'retail', 'Services','Comm. Util.', 'Wholesale','Government']

        # Check for missing columns
        missing_cols1 = [col for col in set1 if col not in merged_df.columns]
        missing_cols2 = [col for col in set2 if col not in merged_df.columns]

        if missing_cols1 or missing_cols2:

          print("Missing columns in set1:", missing_cols1)
          print("Missing columns in set2:", missing_cols2)

        else:

          full_corr_matrix = merged_df[set1 + set2].corr()

          # Extract only the correlation between set1 and set2
          corr_matrix = full_corr_matrix.loc[set1, set2]

          print("\nCorrelation Matrix:\n", corr_matrix)

        # Compute p-values using Pearson correlation test
        p_values = pd.DataFrame(index=set2, columns=set1)

        for col2 in set2:
          for col1 in set1:
             _, p_value = pearsonr(merged_df[col2].values, merged_df[col1].values)
             p_values.loc[col2, col1] = p_value

        print("\nP-value Matrix:\n", p_values)
        """
        ############################################################################

        array = np.zeros((merged_dfs_employ.shape))
        array = merged_dfs_employ.to_numpy()
        #array[:, [-1]] = (array[:, [-1]] - np.min(array[:, [-1]])) / (np.max(array[:, [-1]]) - np.min(array[:, [-1]]))
        array = np.delete(array, [0, 1], axis=0)
        array = np.delete(array, 0, axis=1)

        simulation = self.stationary_block_bootstrap(array, p, maturity, rng)
        simData = np.empty((maturity, array.shape[1] + 1))
        simData.fill(np.nan)
        simData[:, 0:-1] = simulation
        simData[:, -1] = np.tile(np.arange(self.numPeriods + 1, self.numPeriods_retire, -1), self.numSim * numBatches)
        #print(simData.shape)
        #plt.scatter(simData[0:999,[12]],simData[1:1000,[1]], color='blue', marker='o', s=100)

        #independent_vars1 = sm.add_constant(simData[:-1,[10]]) after bootstrapping mkt_dp
        #independent_vars2 = sm.add_constant(simData[:-1,[12]]) after bootstrapping fin_dp

        #model_1 = sm.OLS(simData[1:,[8]], independent_vars1).fit() 为了测试after bootstrapping后的market_dp结果是不是好的predictor对于mkt equity
        #print(model_1.summary())
        #model_2 = sm.OLS(simData[1:,[1]], independent_vars1).fit() 为了测试after bootstrapping后的fin_dp结果是不是好的predictor对于fin equity
        #print(model_2.summary())

        #results = self.acf_compare_pre_post(array[:,9], simData[:,9])  # uses defaults set earlier
        #print("Pre significant lags:",  results["pre"]["signif_lags"])
        #print("Pre max significant:",   results["pre"]["max_signif_lag"])
        #print("Post significant lags:", results["post"]["signif_lags"])
        #print("Post max significant:",  results["post"]["max_signif_lag"])

        return  array, simData

    def bootstrap_data_Ic_9_retire(self, parModel):

        numBatches = parModel["numBatches"]
        expected_block_length = 8
        maturity = self.numPeriods_retire * self.numSim * numBatches  # New fixed number of samples for each bootstrap draw
        p = 1 / expected_block_length
        rng = np.random.default_rng(42)

        drive.mount("/content/drive", force_remount=True)
        file_path = '/content/drive/My Drive/industry_porfolio(new).xlsx'

        df_IP = pd.read_excel(file_path,sheet_name="indus_return",engine="openpyxl")
        df_IP = df_IP.iloc[1:]
        df_IP.columns = ['Year', 'Construction_Ip', 'Finance_Ip', 'Manuf_Ip','Mining_Ip', 'retail_Ip', 'Service_Ip','Comm. Util._Ip', 'wholesale_Ip']
        df_IP = df_IP.map(lambda x: str(x).replace('\xa0', '') if isinstance(x, str) else x)
        #df_IP = df_IP.applymap(lambda x: str(x).replace('\xa0', '') if isinstance(x, str) else x)
        df_IP= df_IP.apply(pd.to_numeric, errors='coerce')
        df_IP.iloc[:, 1:] = df_IP.iloc[:, 1:]/ 100 + 1
        df_IP.iloc[:, 1:] = df_IP.iloc[:, 1:].round(6)

        df_ME = pd.read_excel(file_path,sheet_name="ME",engine="openpyxl")
        df_ME = df_ME.iloc[1:]
        df_ME.columns = ['Year', 'Construction_ME', 'Finance_ME', 'Manuf_ME','Mining_ME', 'retail_ME', 'Service_ME','Comm. Util._ME', 'wholesale_ME']

        # --- Align frames on Year and make Year the index ---
        ip = df_IP.copy().set_index('Year')
        me = df_ME.copy().set_index('Year')

        # Get the common industry base names by stripping the suffixes
        ip_cols = [c.replace('_Ip','') for c in ip.columns]
        me_cols = [c.replace('_ME','') for c in me.columns]
        assert set(ip_cols) == set(me_cols), "Industry sets differ between returns and ME."

        # Reorder columns consistently by the returns order
        industries = ip_cols
        ip = ip[[f"{ind}_Ip" for ind in industries]]
        me = me[[f"{ind}_ME" for ind in industries]]

        # Convenience: put returns and ME in matching wide DataFrames with same column names
        R = ip.copy()
        R.columns = industries
        W = me.copy()
        W.columns = industries

        # --- Compute R_{M/k} for each exclusion k ---
        df_mk = pd.DataFrame(index=R.index)

        for k in industries:
            # exclude industry k
            R_excl = R.drop(columns=k)
            W_excl = W.drop(columns=k)

            # row-wise normalized weights excluding k
            denom = W_excl.sum(axis=1)
            w_norm = W_excl.div(denom, axis=0)

            # weighted average gross return excluding k
            df_mk[f"R_M_excl_{k}"] = (w_norm * R_excl).sum(axis=1)

        # Put Year back as a column (optional)
        df_mk = df_mk.reset_index()

        file_path = '/content/drive/My Drive/project data/F-F_Research_Data_Factors.xlsx'
        df1 = pd.read_excel(file_path, engine="openpyxl")# Try different delimiters if neede
        df1 = df1.iloc[1:]
        df1.columns = ['Year', 'Mkt', 'RF']
        df1['Mkt']=  df1['Mkt']/100 + 1
        df1['RF'] = df1['RF']/100 + 1

        file_path_cpi = '/content/drive/My Drive/project data/CPI.xlsx'
        df_If = pd.read_excel(file_path_cpi, sheet_name="sheet2", engine="openpyxl")
        df_If.columns = ['Year', 'inflation']

#################################################################  just create the predictors
        df_If_lag = df_If.copy()
        df_If_lag['Year'] =  df_If_lag['Year'] + 1

        new_df1_lag = df1[['Year', 'Mkt']].copy()
        new_df1_lag['Year'] = new_df1_lag['Year'] +  1

        df_IP_lag = df_IP.copy()
        df_IP_lag['Year'] = df_IP_lag['Year'] + 1
        df_forecast = [new_df1_lag,df_IP_lag,df_If_lag]

        df_IP_lag = reduce(lambda left, right: pd.merge(left, right, on='Year', how='inner'),df_forecast)

        df_IP_lag.columns = ['Year','Mkt_lag', 'Construction_Ip_lag', 'Finance_Ip_lag', 'Manuf_Ip_lag','Mining_Ip_lag', 'retail_Ip_lag', 'Service_Ip_lag','Comm. Util._Ip_lag', 'wholesale_Ip_lag','inflation']
        select_columns = ['Mkt_lag', 'Construction_Ip_lag', 'Finance_Ip_lag', 'Manuf_Ip_lag','Mining_Ip_lag', 'retail_Ip_lag', 'Service_Ip_lag','Comm. Util._Ip_lag', 'wholesale_Ip_lag']

        df_IP_lag[select_columns] = df_IP_lag[select_columns].apply(
            lambda x: x.values /  df_IP_lag['inflation'].values)
        df_IP_lag.drop(columns=['inflation'], inplace=True)

######################################################################
        #file_path_IC = '/content/drive/My Drive/project data/split_yearlyPerCapitaWages(new).xlsx'
        #df_IC = pd.read_excel(file_path_IC, sheet_name="sheet1", engine="openpyxl")

        columns_IC = ['Year', 'Construction', 'Finance', 'Manufacturing', 'Mining',
              'retail', 'Services', 'Comm. Util.', 'Wholesale', 'Government']
        df_IC = pd.DataFrame(0.0,index=df_IP.index, columns=columns_IC)
        df_IC['Year'] = df_IP['Year']

        file_path_dp = '/content/drive/My Drive/project data/predicor_industry.xlsx'
        df_dp = pd.read_excel(file_path_dp, sheet_name="Sheet1", usecols="A:J", engine="openpyxl")
        df_dp.columns = ["Year", "dp_market", "dp_constr","Fin_dp","dp_manuf","Mine_dp","Retail_dp","Service_dp","Comm. Util_dp","Wholesale_dp"]
        # Reset the index, if necessary
        df_dp = df_dp.reset_index(drop=True)

        df_dp_lag = df_dp.copy()
        df_dp_lag['Year'] = df_dp_lag['Year'] + 1
        df_dp_lag.columns = ["Year", "dp_market_lag", "dp_constr_lag","Fin_dp_lag","dp_manuf_lag","Mine_dp_lag","Retail_dp_lag","Service_dp_lag","Comm. Util_dp_lag","Wholesale_dp_lag"]

        #df_dp['Year'] = pd.to_numeric(df_dp['Year'], errors='coerce')
        #df_IC.iloc[:, 1:] = df_IC.iloc[:, 1:] + 1

        dfs_retire = [df_IP,df_mk,df1, df_dp, df_IC, df_IP_lag,df_dp_lag, df_If]
        merged_dfs_retire= reduce(lambda left, right: pd.merge(left, right, on='Year', how='inner'),dfs_retire)

        select_columns = ['Construction_Ip', 'Finance_Ip', 'Manuf_Ip','Mining_Ip', 'retail_Ip', 'Service_Ip','Comm. Util._Ip', 'wholesale_Ip','Mkt', 'RF','R_M_excl_Construction', 'R_M_excl_Finance', 'R_M_excl_Manuf','R_M_excl_Mining', 'R_M_excl_retail', 'R_M_excl_Service',
                          'R_M_excl_Comm. Util.', 'R_M_excl_wholesale']
        merged_dfs_retire[select_columns] = merged_dfs_retire[select_columns].apply(
            lambda x: x.values / merged_dfs_retire['inflation'].values)

        ################################
        merged_dfs_retire.drop(columns=['inflation'], inplace=True)
        merged_dfs_retire = merged_dfs_retire.iloc[:-1]
        summary_stats = merged_dfs_retire.agg(['mean', 'std', 'skew', 'kurtosis', 'min', 'max', 'count'])
        print(tabulate(summary_stats, headers='keys', tablefmt='pretty'))

        # Extract predictors (current period)
        #dp_mkt = merged_df['dp_market'].to_numpy()[1:]  # t=0 to n-2
        #dp_mkt_lag = merged_df['dp_market'].to_numpy()[:-1]  # t=0 to n-2
        #dp_mkt_lag = np.log(dp_mkt_lag)
        #dp_fin = merged_dfs_retire['Fin_dp'].to_numpy()[1:]  # t=0 to n-2
        #dp_fin_lag = merged_dfs_retire['Fin_dp'].to_numpy()[:-1]  # t=0 to n-2
        #dp_con_lag = np.log(dp_con_lag)

        # Extract outcomes (next period)
        #r_mkt = merged_df['Mkt'].to_numpy()[1:] # t=1 to n-1
        #ex_r_mkt = np.log(r_mkt)- np.log(rf)

        #r_mkt_lag = merged_df['Mkt'].to_numpy()[:-1]
        #rf_lag = merged_df['RF'].to_numpy()[:-1]
        #ex_r_mkt_lag = np.log(r_mkt_lag) - np.log(rf_lag)

        #print(merged_dfs_retire)

        array = np.zeros((merged_dfs_retire.shape))
        array = merged_dfs_retire.to_numpy()
        #array[:, [-1]] = (array[:, [-1]] - np.min(array[:, [-1]])) / (np.max(array[:, [-1]]) - np.min(array[:, [-1]]))
        array = np.delete(array, [0, 1], axis=0)
        array = np.delete(array, 0, axis=1)

        simulation = self.stationary_block_bootstrap(array, p, maturity, rng)
        simData = np.empty((maturity, array.shape[1] + 1))
        simData.fill(np.nan)
        simData[:, 0:-1] = simulation
        simData[:, -1] = np.tile(np.arange(self.numPeriods_retire, 0, -1), self.numSim * numBatches)


        return  simData

    def geometric_block_length(self, p, rng):
        return rng.geometric(p)

    def stationary_block_bootstrap(self, data, p, maturity, rng):
        _, n_assets = data.shape
        bootstrapped_data = np.empty((maturity, n_assets))

        current_idx = 0
        while current_idx < maturity:
            block_length = self.geometric_block_length(p, rng)
            start_point = rng.integers(0, _)
            block_end = start_point + block_length

            if block_end <= _:
                block = data[start_point:block_end, :]
            else:
                block = data[start_point:, :]

                # block = np.concatenate((data[:, start_point:], data[:, 0: block_end - _]), axis=1)

            block_size = block.shape[0]
            if current_idx + block_size > maturity:
                block = block[:maturity - current_idx, :]
                block_size = block.shape[0]

            bootstrapped_data[current_idx:current_idx + block_size, :] = block
            current_idx += block_size

        return bootstrapped_data


    def set_acf_defaults(self, max_lag: int = 10, alpha: float = 0.05, use_fft: bool = False) -> None:

        self.acf_max_lag = int(max_lag)
        self.acf_alpha   = float(alpha)
        self.acf_use_fft = bool(use_fft)


    def acf_significance(self,
                          x: np.ndarray,
                          max_lag: Optional[int] = None,
                          alpha: Optional[float] = None,
                          use_fft: Optional[bool] = None) -> Dict[str, Any]:

        """
        Compute ACF up to max_lag with (1-alpha) pointwise confidence bands
        and report which lags are significantly different from zero.

        Returns:
            {
              'acf': np.ndarray (lags 0..max_lag),
              'confint': np.ndarray shape (max_lag+1, 2),
              'signif_lags': List[int],        # significant lags (exclude lag 0)
              'max_signif_lag': int or None    # largest significant lag
            }
        """
          # use instance defaults if not provided
        if max_lag is None: max_lag = getattr(self, 'acf_max_lag', 10)
        if alpha   is None: alpha   = getattr(self, 'acf_alpha',   0.05)
        if use_fft is None: use_fft = getattr(self, 'acf_use_fft', False)

        acf_vals, confint = acf(x, nlags=max_lag, alpha=alpha, fft=use_fft)

        signif: List[int] = []
        for k in range(1, max_lag + 1):  # skip lag 0
            lo, hi = confint[k]
            if not (lo <= 0.0 <= hi):
                signif.append(k)

        return {
            "acf": acf_vals,
            "confint": confint,
            "signif_lags": signif,
            "max_signif_lag": (max(signif) if signif else None),
        }

    def acf_compare_pre_post(self,
                              pre: np.ndarray,
                              post: np.ndarray,
                              max_lag: Optional[int] = None,
                              alpha: Optional[float] = None,
                              use_fft: Optional[bool] = None) -> Dict[str, Any]:
        """
        Run acf_significance() on two arrays: 'pre-training' and 'after-training'.
            """
        res_pre  = self.acf_significance(pre,  max_lag=max_lag, alpha=alpha, use_fft=use_fft)
        res_post = self.acf_significance(post, max_lag=max_lag, alpha=alpha, use_fft=use_fft)
        return {"pre": res_pre, "post": res_post}


In [ ]:
class investorTf:
    """
    Parameters:

    1. Define the utility function 'utilFunc'
        Possibilities:
            a) 'Power'
            b) 'Exponential'
            c) 'TwoPartPower'

    2. Give the parameters in the utility function
        For 'Power': [gamma, beta]
        For 'Exponential': [a, beta]
        For 'TwoPartPower':  [gammaG, gammaL, threshold, kappa, beta]

    3. Determine wealth of investor
        Net of human capital, i.e., present value of endowments is added below


    """

    def __init__(self, utilFunc = 'Power'):


        ###########
        # define the investor

        self.utilFunc = utilFunc


    ##########################################################################
    ## Call methods
    ##########################################################################

    def utility(self, consume, parUtil):

        return eval("self."+self.utilFunc+"Util")(consume, parUtil)


    def margUtility(self, consume, parUtil):

        return eval("self."+self.utilFunc+"MargUtil")(consume, parUtil)


    def inverseMargUtility(self, util, parUtil):

        return eval("self."+self.utilFunc+"InverseMargUtil")(util, parUtil)

    ##########################################################################
    ## Two-Part Power, varying reference level
    ##########################################################################

    def TwoPartPowerVarUtil(self, consume, parUtil):

        alpha = parUtil[4]
        beta = parUtil[5]

        # Create a tensor of zeros with the same shape as consume
        thresh = tf.zeros(consume.shape, dtype=tf.float32)

        # Set the initial values for the first column
        thresh = tf.tensor_scatter_nd_update(thresh, indices=[tf.constant([tf.range(consume.shape[0]), 0])],
                                             updates=parUtil[2])

        # Define the loop for updating the remaining columns
        for iTime in range(1, consume.shape[1]):
            updates = beta * consume[:, iTime - 1] + (1 - alpha) * thresh[:, iTime - 1]
            indices = [tf.constant([tf.range(consume.shape[0]), iTime])]
            thresh = tf.tensor_scatter_nd_update(thresh, indices=indices, updates=updates)

        idx = tf.cast(consume >= thresh, dtype=tf.float32)

        firstPart = tf.math.pow(tf.math.multiply((consume - thresh), idx) + tf.keras.backend.epsilon(), parUtil[0])
        secondPart = - parUtil[3] * tf.math.pow(tf.math.multiply((thresh - consume), 1.0 - idx) + tf.keras.backend.epsilon(), parUtil[1])

        return tf.math.add(firstPart, secondPart)
        # return firstPart

    ##########################################################################
    ## Two-Part Power
    ##########################################################################

    def TwoPartPowerUtil(self, consume, parUtil):
        thresh = parUtil[2]
        idx = tf.cast(consume >= thresh, dtype=tf.float32)

        firstPart = tf.math.pow(tf.math.multiply((consume - thresh), idx) + tf.keras.backend.epsilon(), parUtil[0])
        secondPart = - parUtil[3] * tf.math.pow(tf.math.multiply((thresh - consume), 1.0-idx) + tf.keras.backend.epsilon(), parUtil[1])

        return tf.math.add(firstPart, secondPart)
        #return firstPart

    def TwoPartPowerMargUtil(self, consume, parUtil):
        thresh = parUtil[2]
        idx = tf.cast(consume >= thresh, dtype=tf.float32)
        tmp1 = tf.math.pow(parUtil[0] * (consume - thresh), parUtil[0] - 1) * idx
        tmp2 = tf.math.pow(parUtil[3] * parUtil[1] * (thresh - consume), parUtil[1] - 1) * (1 - idx)

        return tf.where(tf.math.is_nan(tmp1), 0, tmp1) + \
               tf.where(tf.math.is_nan(tmp2), 0, tmp2)


    # def TwoPartPowerInverseMargUtil(self, util, parUtil):
    #
    #     thresh = parUtil[2]
    #     cons = np.empty((2,1))
    #
    #     cons[0,0] = (util**(1/(parUtil[0]-1)))/parUtil[0] + thresh
    #
    #     cons[1,0] = thresh - (util**(1/(parUtil[1]-1)))/(parUtil[1]*parUtil[3])
    #
    #     return cons

    ##########################################################################
    ## CRRA
    ##########################################################################

    def PowerUtil(self, consume, parUtil):

        if parUtil[0] == 1.0:
            return tf.math.log(consume)
        else:
            return tf.math.pow(consume, 1 - parUtil[0]) / (1.0 - parUtil[0])

    def PowerMargUtil(self, consume, parUtil):

        if parUtil[0] == 1.0:
            return 1 / consume
        else:
            return tf.math.pow(consume, -parUtil[0])

    def PowerInverseMargUtil(self, util, parUtil):

        if parUtil[0] == 1.0:
            return util
        else:
            return tf.math.pow(util, -1.0 / parUtil[0])


In [ ]:
from tensorflow.keras.callbacks import Callback

class PrintIntermediateValuesCallback(Callback):
    def __init__(self, intermediate_model, data, num_periods):
        super().__init__()
        self.intermediate_model = intermediate_model
        self.data = data
        self.numPeriods = num_periods

    def on_batch_end(self, batch, logs=None):

        for batch_data in self.data.take(1):
            inputs = batch_data[0]
            outputs = self.intermediate_model(inputs)
            #debug_tensors["final_consWealthOut"] = consWealthOut
            #print(f"final_consWealthOut:", outputs.shape, "value:", outputs.numpy())
            #for iTime in range(0,self.numPeriods):

              #print(f"Batch {batch} -beforeIncome at iTime {iTime} shape:", outputs[iTime * 8].shape, "value:", outputs[iTime * 8].numpy())
              #print(f"Batch {batch} -beforeWealth at iTime {iTime} shape:", outputs[iTime * 8 +1].shape, "value:", outputs[iTime * 8+1].numpy())
              #print(f"Batch {batch} -consLayer_outputs at iTime {iTime} shape:", outputs[iTime * 8+2].shape, "value:", outputs[iTime * 8+2].numpy())
              #print(f"Batch {batch} -consWealthOut at iTime {iTime} shape:", outputs[iTime * 8+3].shape, "value:", outputs[iTime * 8+3].numpy())
              #print(f"Batch {batch} -currentCons  at iTime {iTime} shape:", outputs[iTime * 8+4].shape, "value:", outputs[iTime * 8+4].numpy())
              #print(f"Batch {batch} -outputWealthOut at iTime {iTime} shape:", outputs[iTime * 8+5].shape, "value:", outputs[iTime * 8+5].numpy())
              #print(f"Batch {batch} - currentIncome at iTime {iTime} shape:", outputs[iTime * 8+6].shape, "value:", outputs[iTime * 8+6].numpy())
              #print(f"Batch {batch} - currentWealth at iTime {iTime} shape:", outputs[iTime * 8+7].shape, "value:", outputs[iTime * 8+ 7].numpy())
            print(f"final_consWealthOut:", outputs[-1].shape, "value:", outputs[-1].numpy())
            print("Type of consWealthOut after the loop:", type(outputs[-1]))


class PrintInputMatrixCallback(Callback):
    def __init__(self, data):
        super().__init__()
        self.data = data

    def on_batch_end(self, batch, logs=None):
        for batch_data in self.data.take(1):
            inputs = batch_data[0]
            print(f"Batch {batch} - input_matrix shape:", inputs.shape)
            print(f"Batch {batch} - input_matrix sample:", inputs.numpy()[0, :5, :])  # Print first sequence, first 2 time steps

In [ ]:
from tensorflow.keras.callbacks import Callback

class PrintIntermediateValuesCallback_new(Callback):
    def __init__(self, intermediate_model, data, num_periods):
        super().__init__()
        self.intermediate_model = intermediate_model
        self.data = data
        self.numPeriods = numPeriods
        self.iterator = iter(self.data)  # Create an iterator for the dataset

    def on_batch_end_2(self, batch, logs=None):
        # Get the current batch's input
        batch_data = next(self.iterator, None)
        if batch_data is None:  # Reset iterator if exhausted
            self.iterator = iter(self.data)
            batch_data = next(self.iterator)
            inputs = batch_data[0]  # Shape: (50, 41, 7)
            outputs = self.intermediate_model(inputs)
            #for iTime in range(0,self.numPeriods):
              #print(f"Batch {batch} -IncomeGrowth at iTime {iTime} shape:", outputs[iTime*3].shape, "value:", outputs[iTime*3].numpy())
              #print(f"Batch {batch} -BeforeIncome at iTime {iTime} shape:", outputs[iTime*3+1].shape, "value:", outputs[iTime*3+1].numpy())
              #print(f"Batch {batch} -AfterIncome at iTime {iTime} shape:", outputs[iTime*3+2].shape, "value:", outputs[iTime*3+2].numpy())
        #print(f"Batch {batch} - input_matrix shape:", inputs.shape)
        #print(f"Batch {batch} - input_matrix sample:", inputs.numpy())
        print(f"final_consWealthOut:", outputs[-1].shape, "value:", outputs[-1].numpy())
        print("Type of consWealthOut after the loop:", type(outputs[-1]))

    def on_batch_end(self, batch, logs=None):

        batch_data = next(self.iterator, None)

        if batch_data is None:
            self.iterator = iter(self.data)
            batch_data = next(self.iterator, None)

            # Still None? Return early to avoid crash
            if batch_data is None:
                print(f"Batch {batch}: Skipped due to empty batch_data.")
                return

        inputs = batch_data[0]  # Example shape: (50, 41, 7)
        outputs = self.intermediate_model(inputs)

        print(f"final_consWealthOut:", outputs[-1].shape, "value:", outputs[-1].numpy())
        print("Type of consWealthOut after the loop:", type(outputs[-1]))


class PrintInputMatrixCallback_new(Callback):
    def __init__(self, data):
        super().__init__()
        self.data = data
        self.iterator = iter(self.data)  # Create an iterator for the dataset

    def on_batch_end(self, batch, logs=None):
        batch_data = next(self.iterator, None)
        if batch_data is None:  # Reset iterator if exhausted
            self.iterator = iter(self.data)
            batch_data = next(self.iterator)
        inputs = batch_data[0]  # Shape: (50, 41, 7)
        #print(f"Batch {batch} - input_matrix shape:", inputs.shape)
        #print(f"Batch {batch} - input_matrix sample:", inputs.numpy()[:5])

In [ ]:
import numpy as np
import pickle
import tensorflow as tf

class PickleCheckpoint(tf.keras.callbacks.Callback):
    """
    A custom callback to save model weights as a pickle file.
    Similar to ModelCheckpoint, but uses pickle instead of HDF5.
    """
    def __init__(
        self,
        filepath,
        monitor='val_loss',
        verbose=0,
        save_best_only=False,
        mode='auto'
    ):
        super().__init__()
        self.filepath = filepath
        self.monitor = monitor
        self.verbose = verbose
        self.save_best_only = save_best_only

        # Determine whether to look for min or max if mode='auto'
        if mode not in ['auto', 'min', 'max']:
            print(f"[PickleCheckpoint] Unknown mode '{mode}', fallback to 'auto'.")
            mode = 'auto'
        self.mode = mode

        if self.mode == 'min':
            self.monitor_op = np.less
            self.best = np.inf
        elif self.mode == 'max':
            self.monitor_op = np.greater
            self.best = -np.inf
        else:
            # auto: guess based on monitor
            if 'acc' in self.monitor or 'accuracy' in self.monitor:
                self.monitor_op = np.greater
                self.best = -np.inf
            else:
                self.monitor_op = np.less
                self.best = np.inf

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        current = logs.get(self.monitor)
        if current is None:
            if self.verbose > 0:
                print(f"[PickleCheckpoint] {self.monitor} not found in logs; skipping save.")
            return

        if not self.save_best_only or self.monitor_op(current, self.best):
            if self.verbose > 0 and self.save_best_only:
                print(f"Epoch {epoch+1}: {self.monitor} improved "
                      f"from {self.best} to {current}, saving weights to {self.filepath}")
            self.best = current

            # Extract weights as a list of NumPy arrays
            weights = self.model.get_weights()

            # Save weights via pickle
            with open(self.filepath, 'wb') as f:
                pickle.dump(weights, f)
        else:
            if self.verbose > 0 and self.save_best_only:
                print(f"Epoch {epoch+1}: {self.monitor} did not improve from {self.best}.")


In [ ]:
class GradientLoggingCallback(tf.keras.callbacks.Callback):
    def on_train_batch_end(self, batch, logs=None):
        x, y = self.validation_data  # this gets injected when we override fit()

        with tf.GradientTape() as tape:
            y_pred = self.model(x, training=True)
            loss = self.model.compiled_loss(y, y_pred, regularization_losses=self.model.losses)

        grads = tape.gradient(loss, self.model.trainable_weights)

        max_grad = tf.reduce_max([tf.reduce_max(tf.abs(g)) for g in grads if g is not None])
        tf.print(f"🔍 Batch {batch} - Max gradient magnitude:", max_grad)


In [ ]:
class GradientLoggingModel(tf.keras.Model):

    def train_step(self, data):
        x, y = data

        with tf.GradientTape() as tape:
            y_pred = self(x, training=True)
            loss = self.compiled_loss(y, y_pred, regularization_losses=self.losses)

        grads = tape.gradient(loss, self.trainable_weights)

        # Log gradient info
        max_grad = tf.reduce_max([tf.reduce_max(tf.abs(g)) for g in grads if g is not None])
        tf.print("🔍 Max gradient magnitude:", max_grad)

        # Apply gradients
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))

        # Standard metric logging
        self.compiled_metrics.update_state(y, y_pred)
        return {m.name: m.result() for m in self.metrics}

In [ ]:
import os, glob
import time
from joblib import Parallel, delayed
import scipy.optimize as spopt
from scipy.optimize import root
from scipy.stats import norm
# from scipy.stats import matrix_normal
# from scipy.optimize import Bounds
from scipy import interpolate
import tensorflow as tf
from tensorflow.keras import backend as K
#import matplotlib.pyplot as plt
import pickle
from tensorflow.keras import layers
from tensorflow.keras.layers import Input, Dense, Concatenate, Lambda
from tensorflow.keras.models import Model
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

class portfolioDiscreteNew:

    def __init__(self, investor, economy, parsSolution):

        tf.keras.backend.set_floatx('float64')
        ###########
        # define the investor by using the class passed
        self.investor = investor

        ###########
        # define the economy by using the class passed
        self.economy = economy

        self.numAssets = self.economy.numAssets  # including the risk-free asset
        self.numStates = self.economy.numStates

        self.numPeriods_employ = self.economy.numPeriods_employ
        self.numPeriods_retire = self.economy.numPeriods_retire
        self.numPeriods = self.numPeriods_employ + self.numPeriods_retire
        self.numSim = self.economy.numSim

        ###########
        # solution method parameters
        #self.maximization = maximization

        ############
        # parameters needed in one dictionary
        self.parsSolution= parsSolution
        #self.numGrid = self.parsSolution["numGrid"]  # number of portfolio grids

    #####################################################################################
    #####################################################################################
    #####################################################################################
    ## methods to compute the value of the criterion function
    ##

    def updateWealthCons(self, input):

        consIn = input[:,0]
        wealthIn = input[:,1]
        weightsIn = input[:,2:(self.numAssets+2)]

        # normalize the weights to sum to one (if they already are, nothing changes in this line)
        weightsIn = weightsIn / tf.reduce_sum(weightsIn, axis=1, keepdims=True)

        returnsIn = tf.transpose(input[:,(self.numAssets+2):])
        portfRet = tf.linalg.diag_part(tf.matmul(weightsIn,returnsIn))

        outTensor = (wealthIn - consIn) * portfRet

        return tf.reshape(outTensor, (-1, 1))


    def neuralNetPolicyConsumLoss(self, y_true, y_pred, parsUtil, adjustFactor):

        # The loss is computed across the sequences in the batch: y_pred is a numSeq x time matrix, the mean is computed across axis 0
        # The "+0.000000001" is there to avoid -infinity outcomes for states in which the wealth goes to zero. Note, that the varoables are in 32 bit.
        batch_loss = -adjustFactor*tf.keras.backend.mean(self.investor.utility(y_pred+0.001, parsUtil), axis=0)

        return 10000*tf.keras.backend.sum(batch_loss)


    def get_checkpoint_filepath(self,process_id):

        #base_dir = '/content/drive/My Drive/without_predictor/ModelCheckpoints/'
        base_dir = '/content/drive/My Drive/without_predictor/ModelCheckpoints/'
        extend2_path = os.path.join(base_dir, str(self.parsSolution["parUtil"][0]), str(self.parsSolution["industry"]),str(f'risk_income_1'))
        os.makedirs(extend2_path, exist_ok=True)

        return os.path.join(extend2_path, f'model_process_{process_id+50}.weights.h5')

    def get_weights_filepath(self, process_id):

        #base_dir = '/content/drive/My Drive/without_predictor/parameters/'
        base_dir = '/content/drive/My Drive/without_predictor/parameters/'
        extend3_path = os.path.join(
            base_dir,
            str(self.parsSolution["parUtil"][0]),
            str(self.parsSolution["industry"]),
            str(f'risk_income_1')
        )
        os.makedirs(extend3_path, exist_ok=True)

        #return os.path.join(extend3_path, f'model_process_{process_id}.weights.h5')
        return os.path.join(extend3_path, f'model_process_{process_id+50}.pkl')


    def policyFunctionConsIncEval(self, modelNn, data):

        numTime = data.shape[0]  #61
        wealthOut = np.empty((numTime+1,1)) #62
        consOut = np.empty((numTime, 1))  #61
        consWealthOut = np.empty((numTime, 1))  #61
        weightsOut = np.empty((numTime-1,self.numAssets))
        current_Wealth = self.parsSolution["startWealth"]
        current_Income = self.parsSolution["startIncome"]
        wealthOut[0, 0] = current_Wealth + current_Income
        current_Wealth = self.parsSolution["startWealth"] + self.parsSolution["startIncome"]

        for iPeriod in range(0, numTime-1): #60

            # Use the model to get the consumption
            iCount = 0
            countLayer = 0
            tmpVal = np.append(data[iPeriod, self.parsSolution["statesStart"]:self.parsSolution["statesEnd"]], current_Wealth/(self.parsSolution["startWealth"]+self.parsSolution["startIncome"]))

            # Hidden layer(s)
            for iLayer in range(0, self.parsSolution["numLayers"]-2):

                #print("First-layer in_dim expected:", modelNn[0].shape[0])
                #print("idata length now:", np.asarray(tmpVal[0]).size)

                tmpVal = tmpVal @ modelNn[iCount] + modelNn[iCount + 1]
                tmpVal = eval("self."+self.parsSolution["activation"][countLayer])(tmpVal)
                countLayer+=1
                iCount += 2

            # write the consumption into the output array
            tmpValCons = tmpVal @ modelNn[iCount] + modelNn[iCount + 1]
            consWealthOut[iPeriod, :] = eval("self."+self.parsSolution["activation"][countLayer+1])(tmpValCons)
            consOut[iPeriod, :] = consWealthOut[iPeriod, :] * wealthOut[iPeriod,0]
            iCount += 2

            # write the weights into the output array
            tmpValPort = tmpVal @ modelNn[iCount] + modelNn[iCount + 1]
            tmpPortfolio = eval("self."+self.parsSolution["activation"][countLayer])(tmpValPort)
            weightsOut[iPeriod, :] = tmpPortfolio / np.sum(tmpPortfolio)

            # Update the value for the wealth using the weights
            current_Wealth = (current_Wealth - consOut[iPeriod, :]) * (data[iPeriod + 1, 0:self.numAssets] @ weightsOut[iPeriod, :]) if iPeriod <=numTime-2 else current_Wealth - consOut[iPeriod, :]
            current_Income =  current_Income * data[iPeriod + 1, -1] if iPeriod <= self.numPeriods_employ-1 else 0

            current_Wealth = current_Wealth + current_Income if iPeriod <=numTime-2  else current_Wealth
            # write the wealth into the output array
            wealthOut[iPeriod+1, 0] =  current_Wealth

        return (weightsOut, wealthOut, consOut, consWealthOut)

    def policyFunctionConsIncEval_snap1(self, modelNn, data, numTime):

        wealthOut = np.empty((numTime+1,1))
        consOut = np.empty(numTime)
        consWealthOut = np.empty((numTime, 1))
        weightsOut = np.empty((numTime-1,self.numAssets))
        current_Wealth = self.parsSolution["startWealth"]
        current_Income = self.parsSolution["startIncome"]
        wealthOut[0, 0] = current_Wealth + current_Income
        current_Wealth = self.parsSolution["startWealth"] + self.parsSolution["startIncome"]

        for iPeriod in range(0,numTime-1):

            # Use the model to get the consumption
            iCount = 0
            countLayer = 0
            current_Wealth = current_Wealth + current_Income
            tmpVal = np.append(data[iPeriod, self.parsSolution["statesStart"]:self.parsSolution["statesEnd"]], current_Wealth/(self.parsSolution["startWealth"]+self.parsSolution["startIncome"]))

            # Hidden layer(s)
            for iLayer in range(0, self.parsSolution["numLayers"]-2):
                tmpVal = tmpVal @ modelNn[iCount] + modelNn[iCount + 1]
                tmpVal = eval("self."+self.parsSolution["activation"][countLayer])(tmpVal)
                countLayer+=1
                iCount += 2

            # write the consumption into the output array
            tmpValCons = tmpVal @ modelNn[iCount] + modelNn[iCount + 1]
            consWealthOut[iPeriod, :] = eval("self."+self.parsSolution["activation"][countLayer+1])(tmpValCons)
            consOut[iPeriod] = consWealthOut[iPeriod, :] * wealthOut[iPeriod,0]
            iCount += 2

            # write the weights into the output array
            tmpValPort = tmpVal @ modelNn[iCount] + modelNn[iCount + 1]
            tmpPortfolio = eval("self."+self.parsSolution["activation"][countLayer])(tmpValPort)
            weightsOut[iPeriod,:]= tmpPortfolio / np.sum(tmpPortfolio)

            # Update the value for the wealth using the weights
            current_Wealth = (current_Wealth - consOut[iPeriod]) * (data[iPeriod + 1, 0:self.numAssets] @ weightsOut[iPeriod, :]) if iPeriod <=numTime-2 else current_Wealth - consOut[iPeriod, :]
            current_Income =  current_Income * data[iPeriod + 1, -1] if iPeriod <= self.numPeriods_employ-1 else 0

            current_Wealth = current_Wealth + current_Income if iPeriod <=numTime-2  else current_Wealth
            # write the wealth into the output array
            wealthOut[iPeriod+1, 0] =  current_Wealth

        return current_Wealth

    def policyFunctionConsIncEval_snap2(self, modelNn, data, numTime, index_1):

        index_2 = 0 if index_1 == 4 else 1
        current_Wealth = self.policyFunctionConsIncEval_snap1(modelNn, data, numTime)
        tmpVal = np.append(data[numTime-1, self.parsSolution["statesStart"]:self.parsSolution["statesEnd"]], current_Wealth/(self.parsSolution["startWealth"]+self.parsSolution["startIncome"]))
        num_points = 50
        x_values = np.linspace(0.0,0.06, num_points)
        y_values = []

        for x in x_values:
            modified_mfp = tmpVal.copy()  # Keep other elements unchanged
            modified_mfp[index_1] = x  # Modify only the 5th element

            iCount = 0
            countLayer = 0

            for iLayer in range(0, self.parsSolution["numLayers"]-2):
                modified_mfp = modified_mfp @ modelNn[iCount] + modelNn[iCount + 1]
                modified_mfp = eval("self."+self.parsSolution["activation"][countLayer])(modified_mfp)
                countLayer+=1
                iCount += 2

            iCount += 2

            # write the weights into the output array
            tmpValPort = modified_mfp @ modelNn[iCount] + modelNn[iCount + 1]
            tmpPortfolio = eval("self."+self.parsSolution["activation"][countLayer])(tmpValPort)
            weightsOut= tmpPortfolio / np.sum(tmpPortfolio)
            # Directly access the 2nd element from weightsOut (since it's 1D)
            y_values.append(weightsOut[index_2])

        return y_values


    def eval_equity_weight_from_nn(self, modelNn, y_ratio, x_ratio, z_state_vec, itime, equity_index=0):

        # Use the model to get the consumption

        tmpVal = np.append(z_state_vec, y_ratio)[None, :]
        tmpVal[:,3] = x_ratio

        iCount = 0
        countLayer = 0
        for _ in range(0, self.parsSolution["numLayers"] - 2):

            W = modelNn[iCount]; b = modelNn[iCount + 1]
            tmpVal = tmpVal @ W + b
            tmpVal = eval("self."+self.parsSolution["activation"][countLayer])(tmpVal)
            iCount += 2; countLayer += 1
        Wc = modelNn[iCount]; bc = modelNn[iCount + 1]
        _ = eval("self."+self.parsSolution["activation"][countLayer])(tmpVal @ Wc + bc)
        iCount += 2
        Wp = modelNn[iCount]; bp = modelNn[iCount + 1]
        port_raw = eval("self."+self.parsSolution["activation"][countLayer])(tmpVal @ Wp + bp)
        weights = port_raw / np.sum(port_raw)

        return float(weights[:,equity_index])

    def plot_policy_surfaces_with_state_means(self, modelNn, times, policy_adapter,data, z_grid, x_grid, title_prefix="Policy surface", figsize=(8, 6)):

        Z, X = np.meshgrid(z_grid, x_grid, indexing="xy")

        for spec in times:

            s0 = self.parsSolution["statesStart"]; s1 = self.parsSolution["statesEnd"];
            state_slice = data[:,[spec], s0:s1]
            means = np.mean(state_slice, axis=0)

            W = np.zeros_like(Z, dtype=float)
            for i in range(X.shape[0]):
                for j in range(X.shape[1]):
                    z = Z[i,j]; x = X[i,j];
                    W[i, j] = policy_adapter(modelNn, z, x, means, spec)

            """
            fig = plt.figure(figsize=figsize)
            ax = fig.add_subplot(111, projection="3d")
            ax.plot_surface(X, Z, W, rstride=1, cstride=1, linewidth=0.1, antialiased=True)
            ax.set_xlabel("x = wealth / annual earnings")
            ax.set_ylabel("z = log dp (or percentile)")
            ax.set_zlabel("w = equity share")
            ax.set_title(f"{title_prefix}")
            plt.tight_layout(); plt.show()
            """
            fig = plt.figure(figsize=figsize)
            ax = fig.add_subplot(111, projection="3d")

            surf = ax.plot_surface(
                X, Z, W,
                rstride=1, cstride=1,
                linewidth=0,              # no wireframe edges
                antialiased=True,
                cmap="viridis"            # smooth gradient like your ref figure
            )

            fig.colorbar(surf, ax=ax, shrink=0.6, aspect=14, pad=0.08, label="w = equity share")
            ax.set_box_aspect((1, 1, 0.6))     # nicer proportions
            ax.view_init(elev=28, azim=-55)    # similar viewing angle
            ax.set_xlabel("x = dp ")
            ax.set_ylabel("z = wealth / initial earnings")
            ax.set_zlabel("w = equity share")
            ax.set_title(f"{title_prefix} — t = {spec}")
            #plt.tight_layout(); plt.show()


        plt.tight_layout()
        plt.savefig(title_prefix,dpi=300, bbox_inches="tight")   # <-- one file to download
        plt.show()


    def plot_policy_surfaces_3in1(self, modelNn, times, policy_adapter, data, z_grid, x_grid,
                              fname="policy_surfaces_3in1.png",fix_limits=True):

    # build mesh (keep your current convention)
        Z, X = np.meshgrid(z_grid, x_grid, indexing="xy")

        s0 = self.parsSolution["statesStart"]; s1 = self.parsSolution["statesEnd"]

        fig, axes = plt.subplots(1, 3, figsize=(18, 6), subplot_kw={"projection": "3d"})

        for ax, spec in zip(axes, times):

            # mean other states at time=spec  -> (K,)
            state_slice = data[:, [spec], s0:s1]          # (N,1,K)
            means = np.nanmean(state_slice, axis=(0,1))   # (K,)

            # fill surface
            W = np.zeros_like(Z, dtype=float)
            for i in range(X.shape[0]):
                for j in range(X.shape[1]):
                    z = Z[i, j]; x = X[i, j]
                    W[i, j] = policy_adapter(modelNn, z, x, means, spec)


            # pretty but simple surface (no colorbar)
            ax.plot_surface(X, Z, W, rstride=1, cstride=1, linewidth=0, antialiased=True, cmap="viridis")

            if fix_limits:
               ax.set_xlim(float(x_grid.min()), float(x_grid.max()))
               ax.set_ylim(float(z_grid.min()), float(z_grid.max()))
               ax.set_zlim(0.0, 1.0)

            ax.set_xlabel("x = dp")
            ax.set_ylabel("z = wealth / initial wealth")
            ax.set_zlabel("w = equity share")
            ax.set_title(f"t = {spec}")
            ax.set_box_aspect((1, 1, 0.6))
            ax.view_init(elev=28, azim=-55)

        plt.tight_layout()
        plt.savefig(fname, dpi=300, bbox_inches="tight")   # <-- one file to download
        plt.show()


    def optPortfConsIncNeuralNetPolicy(self,process_id,result):

        try:

            print("Running code for the solution using a policy function determined by a neural network: Consumption and portfolio decision with income")
            print("The setup is:")
            print("Model: {}".format(self.parsSolution['model']))
            print("Utility: {}".format(self.investor.utilFunc))
            print(f"Process {process_id} (PID {os.getpid()}) started.")

            # ensure reproducible neural network fit
            # can be turned off if convergence of algorithm is ensured
            if self.parsSolution["seedIndicator"] == 1:
                tf.random.set_seed(self.parsSolution["seed"])

            # simulate the economy forward to get training and validation data
            if self.parsSolution["simDataPolicyPass"] == None:
                simData = self.economy.simulation(self.parsSolution)
            else:
                simData = self.parsSolution["simDataPolicy"]

            numVars = simData.shape[1]
            numPeriodsSimTrain = self.parsSolution["numBatchesTrain"] * self.numSim * (self.numPeriods + 1)

            numPeriodsSimValidate = (self.parsSolution["numBatchesTrain"] + self.parsSolution[
                "numBatchesVal"]) * self.numSim * (self.numPeriods + 1)


            # Normalize the data using the moments from the training sample: No, since we need the original magnitudes to update the wealth
            # mean = simData[:numPeriodsSimTrain].mean(axis=0)
            # simData -= mean
            # std = simData[:numPeriodsSimTrain].std(axis=0)
            # simData /= std

            # Set up the dataset for the estimation
            dataTrain = tf.keras.utils.timeseries_dataset_from_array(simData[:numPeriodsSimTrain,:], targets=simData[0:numPeriodsSimTrain,1],
                                                                    sequence_length=self.numPeriods+1, sequence_stride=self.numPeriods + 1,
                                                                    batch_size=self.numSim)
            dataValid = tf.keras.utils.timeseries_dataset_from_array(simData[numPeriodsSimTrain:numPeriodsSimValidate, :], targets=simData[numPeriodsSimTrain:numPeriodsSimValidate, 1],
                                                                    sequence_length=self.numPeriods+1,
                                                                    sequence_stride=self.numPeriods + 1,
                                                                    batch_size=self.numSim)
            # get real data
            #realData = self.parsSolution["dataStates"]

            ####################################################
            # Fit the neural network to determine the policy function
            # Define list to store output of layer 2 at each time step (portfolio weights for each time step)
            consumption_outputsAll = []

            # Define the shapes of the model inputs
            # NOTE: The first dimension wil be set to 1, since it is a placeholder for the size of the batch
            input_matrix = tf.keras.layers.Input(shape=(self.numPeriods+1, numVars),dtype='float32', name='input_matrix')


            # Define new tensor with shape (batch_size, 1) and set elements to startWealth
            # NOTE: This ensures that the input tensor will have the same number of rows as elements in the batch
            #       Needs to be flexible (None at the beginning) to be able to adjust the input sizes for model prediction
            input_wealth = self.parsSolution["startWealth"]*tf.keras.backend.ones((tf.keras.backend.shape(input_matrix)[0], 1))

            # Define the layers of the network

            # layer 1 (hidden layer for both decisions)
            hiddenLayer = tf.keras.layers.Dense(self.parsSolution["numNodes"][0], activation=self.parsSolution["activation"][0],
                                          kernel_initializer=tf.keras.initializers.RandomNormal(mean=self.parsSolution["kernelInitializers"][0][0],
                                                                                                stddev=self.parsSolution["kernelInitializers"][0][1]), # 0.0, 0.2
                                          bias_initializer=tf.keras.initializers.Constant(value=self.parsSolution["biasInitializers"][0][0]))

            # If portfolio weight should sum to one use a softmax layer here.
            # layer 2 (portfolio weight layer)
            portfolioLayer = tf.keras.layers.Dense(self.parsSolution["numNodes"][1], activation=self.parsSolution["activation"][1],
                                          kernel_initializer=tf.keras.initializers.RandomNormal(mean=self.parsSolution["kernelInitializers"][1][0],
                                                                                                stddev=self.parsSolution["kernelInitializers"][1][1]),  # 0.0, 0.2
                                          bias_initializer=tf.keras.initializers.Constant(value=self.parsSolution["biasInitializers"][1][0]))

            # layer 3 (neurons for consumption/wealth ratio layer)
            # layer3 = tf.keras.layers.Dense(self.parsSolution["numNodes"][2], activation=self.parsSolution["activation"][2],
            #                                kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.8),
            #                                bias_initializer=tf.keras.initializers.Zeros())

            # layer 3 (consumption/wealth ratio layer)
            consLayer = tf.keras.layers.Dense(self.parsSolution["numNodes"][2], activation=self.parsSolution["activation"][2],
                                          kernel_initializer=tf.keras.initializers.RandomNormal(mean=self.parsSolution["kernelInitializers"][2][0],
                                                                                                stddev=self.parsSolution["kernelInitializers"][2][1]),  # 0.0, 0.5
                                          bias_initializer=tf.keras.initializers.Constant(value=-self.parsSolution["biasInitializers"][2][0])) # -1.0

            # Set up a tensor for the current value of the wealth
            currentWealth = input_wealth
            consWealthOut = tf.keras.backend.zeros((tf.keras.backend.shape(input_matrix)[0], input_matrix.shape[1]-1))

            # Subjective discount factor adjustments
            adjustFactor = (tf.pow(np.array(self.parsSolution["parUtil"][-1]), tf.cast(tf.range(0, self.numPeriods+1), dtype=tf.float32)))

            # Set up loop for recursive network architecture
            # input_matrix.shape[1]-1 = self.numPeriods, since we need the starting value for the states
            # decisions are made at the end of the period where the states are collected, period numPeriods+1 is our first observation =>
            # the end of this period is numPeriods to maturity
            currentIncome =  self.parsSolution["startIncome"]*tf.keras.backend.ones((tf.keras.backend.shape(input_matrix)[0], 1))
            # for the bootstrap case where the income is monthly income return
            currentWealth = tf.math.add(currentWealth, currentIncome)

            for iTime in range(0, input_matrix.shape[1]-1):

                # Include the income into the wealth
                #currentWealth = tf.math.add(currentWealth, tf.math.multiply(input_matrix[:,iTime,-2:-1], self.parsSolution["startIncome"]))
                #income = tf.math.multiply(tf.expand_dims(input_matrix[:, iTime, -1], axis=-1), income)
                #for the case where the income is exactly number instead of retio of initial wealth

                # layer 1 (neurons for portfolio weight layer; normalize the wealth by the start wealth)

                hiddenLayer_input = tf.keras.layers.concatenate([
                    input_matrix[:,iTime, self.parsSolution["statesStart"]:self.parsSolution["statesEnd"]], currentWealth/(self.parsSolution["startWealth"]+ self.parsSolution["startIncome"])])
                hiddenLayer_outputs = hiddenLayer(hiddenLayer_input)

                # If portfolio weight should sum to one use a softmax layer here.
                # layer 2 (portfolio weight layer)
                portfolioLayer_outputs = portfolioLayer(hiddenLayer_outputs)

                # layer 3 (neurons for consumption/wealth ratio layer; uses the same inputs as layer 1)
                # layer3_outputs = layer3(layer1_input)

                # layer 2 (consumption/wealth ratio layer)
                consLayer_outputs = consLayer(hiddenLayer_outputs)
                currentCons = currentWealth*consLayer_outputs
                indices = tf.keras.backend.expand_dims(tf.range(tf.keras.backend.shape(consWealthOut)[0]), axis=-1)
                indices = tf.keras.backend.concatenate([indices, tf.keras.backend.ones_like(indices) * iTime], axis=-1)
                consWealthOut = tf.tensor_scatter_nd_update(consWealthOut, indices, tf.keras.backend.flatten(currentCons))
                #consWealthOut = tf.keras.layers.concatenate([consWealthOut, currentCons])

                # layer 3 (update wealth)
                updateWealth_input = tf.keras.layers.concatenate([currentCons, currentWealth, portfolioLayer_outputs, input_matrix[:,iTime+1, 0:self.numAssets]])
                outputWealthOut = tf.keras.layers.Lambda(lambda x: self.updateWealthCons(x))(updateWealth_input)
                currentIncome = tf.math.multiply(tf.expand_dims(input_matrix[:, iTime+1, -1], axis=-1), currentIncome)
                currentWealth = outputWealthOut + currentIncome

            # Concatenate outputs that collects all the intermediate consumption levels and the final wealth level
            consWealthOut = tf.keras.layers.concatenate([consWealthOut, currentWealth])

            model = tf.keras.Model(inputs=input_matrix, outputs=consWealthOut)
            opt = tf.keras.optimizers.Adam(self.parsSolution["learningRate"], clipnorm=1.0)


            model.compile(
                loss= lambda y_true, y_pred: self.neuralNetPolicyConsumLoss(y_true, y_pred, self.parsSolution["parUtil"], adjustFactor),
                optimizer=opt,
                #metrics=lambda y_true, y_pred: self.neuralNetPolicyLoss(y_true, y_pred, self.parsSolution["parUtil"]),
            )

            # check the model by uncommenting below
            #model.summary()

            # define callbacks
            callbackList = [tf.keras.callbacks.EarlyStopping(monitor='val_loss',
                                                             min_delta=self.parsSolution["min_delta"],
                                                            patience=self.parsSolution["parPatience"]),

                             tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_delta=self.parsSolution["min_delta"],min_lr=1e-6, verbose=1),

                            tf.keras.callbacks.ModelCheckpoint(filepath=self.get_checkpoint_filepath(process_id), monitor="val_loss", save_best_only=True,
                                                              save_weights_only=True)
                            ]

            # fit model
            history = model.fit(
                epochs=self.parsSolution["numEpochs"],
                callbacks=callbackList,
                x=dataTrain,
                validation_data = dataValid
            )

            # load the best model saved in the callback
            #model.load_weights(os.getcwd() + '/my_best_model.h5')

            min_val_loss = min(history.history['val_loss'])

            # Get the list containing the model parameters for the two trainable layers
            #
            # First layer (parameters):
            # The first element has dimension of numVars + 1 by numNodes; + 1 is wealth
            # The second element has dimension numNodes
            #
            # Second layer (weights):
            # The third element has dimension numNodes by numAssets
            # The fourth element has dimension numAssets

            print("Model training is done")
            modelPars = model.get_weights()
            weights_filepath = self.get_weights_filepath(process_id)
            with open(weights_filepath, 'wb') as f:
              pickle.dump(modelPars, f)

            results[process_id] = {
            'min_val_loss': min_val_loss,
            'weights': model.get_weights()  # Save model weights
            }

            print(f"Process {process_id}: Min validation loss = {min_val_loss}")

        except Exception as e:

          print(f"Process {process_id} failed with error: {e}")


    def panacchi_compute_normalized_weights(self):

      simData = self.parsSolution["simDataPolicy"]
      numPeriodsSimTrain = self.parsSolution["numBatchesTrain"] * self.numSim * (self.numPeriods + 1)
      simData = simData[:numPeriodsSimTrain,0:26]

      # Separate risky and risk-free asset returns
      risky_returns = simData[:, :-1]  # All but the last column
      risk_free_returns = simData[:, [-1]]  # Last column
      excess_return = risky_returns - risk_free_returns

      # Compute statistics for risky assets
      mu = excess_return.mean(axis=0)  # Mean return for each risky asset
      sigma = np.cov(risky_returns.T) # Standard deviation for each risky asset
      sigma_inv = np.linalg.inv(sigma)

      # Risk-free rate (assume constant across observations)
      risk_free_rate = risk_free_returns.mean()

      # Compute optimal weights using Merton's formula
      optimal_weights = np.dot(sigma_inv, mu) / self.parsSolution["parUtil"][0]
      risk_free_weight = 1 - np.mean(optimal_weights)
      #optimal_weights = np.append(optimal_weights, risk_free_weight)
      #min_optimal_weights = np.min(optimal_weights)
      #adj_optimal_weights = optimal_weights + abs(min_optimal_weights)

     # Renormalize to ensure the sum equals 1
      #normalized_optimal_weights =  adj_optimal_weights / np.sum(adj_optimal_weights)
      #normalized_optimal_weights = normalized_optimal_weights[:-1]

      consWealthRatio = np.zeros((self.numPeriods+ 1, 1))

      a1Iid = 0

      Ret = np.sum(excess_return * optimal_weights, axis=1) # + risk_free_returns
      Ret = np.reshape(Ret,(-1,1))+risk_free_returns
      PortfRet_temp = np.mean(Ret ** self.parsSolution["parUtil"][0],axis=0)
      a1Iid = (self.parsSolution["parUtil"][1] * PortfRet_temp) ** (-1.0 /(self.parsSolution["parUtil"][0]-1))

      # For the final time point it is one
      consWealthRatio[numPeriods, 0] = 1.0
      for iTime in range(numPeriods, 0, -1):
        consWealthRatio[iTime - 1, 0] = (a1Iid * consWealthRatio[iTime, 0]) / (1 + a1Iid * consWealthRatio[iTime, 0])

      return optimal_weights , consWealthRatio

    def naive_compute_normalized_weights(self):

      simData = self.parsSolution["simDataPolicy"]
      numPeriodsSimTrain = self.parsSolution["numBatchesTrain"] * self.numSim * (self.numPeriods + 1)
      simData = simData[:numPeriodsSimTrain,0:26]

      # Separate risky and risk-free asset returns
      risky_returns = simData[:, :-1]  # All but the last column
      risk_free_returns = simData[:, [-1]]  # Last column
      excess_return = risky_returns - risk_free_returns

      normalized_optimal_weights =  np.array([1/26] * 25)

      consWealthRatio = np.zeros((self.numPeriods+ 1, 1))

      a1Iid = 0

      Ret = np.sum(excess_return * normalized_optimal_weights, axis=1)
      Ret = np.reshape(Ret,(-1,1)) + risk_free_returns
      PortfRet_temp = np.mean(Ret ** self.parsSolution["parUtil"][0],axis=0)
      a1Iid = (self.parsSolution["parUtil"][1] * PortfRet_temp) ** (-1.0 /(self.parsSolution["parUtil"][0]-1))

      # For the final time point it is one
      consWealthRatio[numPeriods, 0] = 1.0
      for iTime in range(numPeriods, 0, -1):
        consWealthRatio[iTime - 1, 0] = (a1Iid * consWealthRatio[iTime, 0]) / (1 + a1Iid * consWealthRatio[iTime, 0])

      return  normalized_optimal_weights, consWealthRatio


    def optPortfConsIncNeuralNetPolicy_normal_2(self):

        print("Running code for the solution using a policy function determined by a neural network: Consumption and portfolio decision with income")
        print("The setup is:")
        print("Model: {}".format(self.parsSolution['model']))
        print("Utility: {}".format(self.investor.utilFunc))

        # Define base directory for checkpoints
        base_dir = '/content/drive/My Drive/ModelCheckpoints/Ex1/'
        os.makedirs(base_dir, exist_ok=True)
        checkpoint_filepath = os.path.join(base_dir, "model.weights.h5")  # Fix filepath to end with .weights.h5

        # Ensure reproducible neural network fit
        if self.parsSolution.get("seedIndicator", 0) == 1:
            tf.random.set_seed(self.parsSolution["seed"])

        # Simulate the economy forward to get training and validation data
        if self.parsSolution.get("simDataPolicyPass") is None:
            simData = self.economy.simulation(self.parsSolution)
        else:
            simData = self.parsSolution["simDataPolicy"]

        numVars = simData.shape[1]
        numPeriodsSimTrain = self.parsSolution["numBatchesTrain"] * self.numSim * (self.numPeriods + 1)
        numPeriodsSimValidate = (self.parsSolution["numBatchesTrain"] + self.parsSolution["numBatchesVal"]) * self.numSim * (self.numPeriods + 1)

        dataTrain = keras.utils.timeseries_dataset_from_array(
        simData[:numPeriodsSimTrain, :],
        targets=simData[:numPeriodsSimTrain, 1],
        sequence_length=self.numPeriods + 1,
        sequence_stride=self.numPeriods + 1,
        batch_size=self.numSim  # Ensure no partial batches
        )

        dataValid = keras.utils.timeseries_dataset_from_array(
            simData[numPeriodsSimTrain:numPeriodsSimValidate, :],
            targets=simData[numPeriodsSimTrain:numPeriodsSimValidate, 1],
            sequence_length=self.numPeriods + 1,
            sequence_stride=self.numPeriods + 1,
            batch_size=self.numSim
        )

        # Debug dataset shape
        #for batch in dataTrain.take(1):
            #print("dataTrain batch shape:", batch[0].shape)  # Should be (500, self.numPeriods + 1, numVars)
        #for batch in dataValid.take(1):
            #print("dataValid batch shape:", batch[0].shape)

        # Define the model using Functional API
        input_matrix = Input(shape=(self.numPeriods + 1, numVars), dtype='float64', name='input_matrix')

        #input_wealth = Lambda(lambda x: tf.cast(self.parsSolution["startWealth"], tf.float64) * tf.ones((tf.shape(x)[0], 1)))(input_matrix)

        input_wealth = Lambda(lambda x: tf.cast(self.parsSolution["startWealth"], tf.float64) * tf.ones((tf.shape(x)[0], 1), dtype=tf.float64),
          output_shape=(1,),dtype='float64')(input_matrix)

        # Define layers
        hiddenLayer_1 = tf.keras.layers.Dense(
            units=self.parsSolution["numNodes"][0],
            activation=self.parsSolution["activation"][0],
            kernel_initializer=tf.keras.initializers.RandomNormal(
                mean=self.parsSolution["kernelInitializers"][0][0],
                stddev=self.parsSolution["kernelInitializers"][0][1]
            ),
            bias_initializer=tf.keras.initializers.Constant(
                value=self.parsSolution["biasInitializers"][0][0]
            )
        )

        hiddenLayer_2 = tf.keras.layers.Dense(
            units=self.parsSolution["numNodes"][1],
            activation=self.parsSolution["activation"][1],
            kernel_initializer=tf.keras.initializers.RandomNormal(
                mean=self.parsSolution["kernelInitializers"][1][0],
                stddev=self.parsSolution["kernelInitializers"][1][1]
            ),
            bias_initializer=tf.keras.initializers.Constant(
                value=self.parsSolution["biasInitializers"][1][0]
            )
        )

        portfolioLayer = tf.keras.layers.Dense(
            units=self.parsSolution["numNodes"][2],
            #activation=self.parsSolution["activation"][2],
            activation= self.custom_activation,
            kernel_initializer=tf.keras.initializers.RandomNormal(
                mean=self.parsSolution["kernelInitializers"][2][0],
                stddev=self.parsSolution["kernelInitializers"][2][1]
            ),
            bias_initializer=tf.keras.initializers.Constant(
                value=-self.parsSolution["biasInitializers"][2][0]
            )
        )

        consLayer = tf.keras.layers.Dense(
            units=self.parsSolution["numNodes"][3],
            activation=self.parsSolution["activation"][3],
            kernel_initializer=tf.keras.initializers.RandomNormal(
                mean=self.parsSolution["kernelInitializers"][3][0],
                stddev=self.parsSolution["kernelInitializers"][3][1]
            ),
            bias_initializer=tf.keras.initializers.Constant(
                value=-self.parsSolution["biasInitializers"][3][0]
            )
        )

        consWealthOut = Lambda(lambda x: tf.zeros((tf.shape(x)[0], tf.shape(x)[1]-1), dtype=tf.float64))(input_matrix)
        currentWealth = input_wealth

        #consWealthOut = tf.keras.backend.zeros((tf.keras.backend.shape(input_matrix)[0], input_matrix.shape[1]-1))

        currentIncome = Lambda(lambda x: tf.cast(self.parsSolution["startIncome"], tf.float64) * tf.ones((tf.shape(x)[0], 1), dtype=tf.float64))(input_matrix)
        #currentWealth = Lambda(lambda inputs: inputs[0] + inputs[1])([currentWealth, currentIncome])
        currentWealth = currentWealth + currentIncome

        debug_tensors = {}
        # Subjective discount factor adjustments
        #adjustFactor = (tf.pow(np.array(self.parsSolution["parUtil"][-1]), tf.cast(tf.range(0, self.numPeriods+1), dtype=tf.float32)))
        # Define adjustFactor as a TensorFlow constant (no need for Lambda)

        adjustFactor = tf.pow(tf.constant(self.parsSolution["parUtil"][-1], dtype=tf.float64),tf.range(0, self.numPeriods+1, dtype=tf.float64))

        # Recursive loop over time step
        for iTime in range(0, input_matrix.shape[1]-1):  #60

        #for iTime in range(0, input_matrix.shape[1]-1):
          #debug_tensors[f"beforecurrrentIncome_{iTime}"] = currentIncome
          #debug_tensors[f"beforecurrrentWealth_{iTime}"] = currentWealth

          hiddenLayer_input = tf.keras.layers.concatenate([
              input_matrix[:,iTime, self.parsSolution["statesStart"]:self.parsSolution["statesEnd"]], currentWealth/(self.parsSolution["startWealth"]+ self.parsSolution["startIncome"])])

          hiddenLayer_outputs_1 = hiddenLayer_1(hiddenLayer_input)
          hiddenLayer_outputs_2 = hiddenLayer_2(hiddenLayer_outputs_1)

          # Layer 2: Portfolio weights
          portfolioLayer_outputs = portfolioLayer(hiddenLayer_outputs_2)

          # Layer 3: Consumption/wealth ratio
          consLayer_outputs = consLayer(hiddenLayer_outputs_2)
          #debug_tensors[f"consLayer_outputs_{iTime}"] = consLayer_outputs
          currentCons = currentWealth*consLayer_outputs

          #batch_size = Lambda(lambda x: tf.shape(x)[0])(consWealthOut) # Compute indices safely
          #indices = Lambda(lambda x: tf.expand_dims(tf.range(x), axis=-1))(batch_size)# Ensure `indices` and `iTime` are compatible
          #iTime_tensor = Lambda(lambda x: tf.ones_like(x) * iTime)(indices)# Concatenate indices properly
          #flattened_currentCons = Lambda(lambda x: tf.reshape(x, [-1]))(currentCons)
          # Compute indices dynamically for the current iTime

          def update_cons_wealth_out(args):
              cons_wealth_out, current_cons, i_time = args
              batch_size = tf.shape(cons_wealth_out)[0]
              batch_indices = tf.range(batch_size, dtype=tf.int64)
              i_time_tensor = tf.ones_like(batch_indices, dtype=tf.int64) * i_time
              indices = tf.stack([batch_indices, i_time_tensor], axis=-1)
              flattened_current_cons = tf.reshape(current_cons, [-1])
              return  tf.tensor_scatter_nd_update(cons_wealth_out, indices, flattened_current_cons)

          preWealthOut = consWealthOut
          consWealthOut  = Lambda(update_cons_wealth_out, output_shape=lambda input_shapes: input_shapes[0])(
              [preWealthOut, currentCons, tf.constant(iTime, dtype=tf.int64)]
          )

          if iTime == input_matrix.shape[1]-1:  #
            print("Condition met, breaking loop.")
            break

          #prev_consWealthOut = consWealthOut
          #consWealthOut = Lambda(lambda x: tf.tensor_scatter_nd_update(x[0], x[1], x[2]),output_shape=lambda input_shapes: input_shapes[0])([consWealthOut, indices, flattened_currentCons])

          updateWealth_input = tf.keras.layers.concatenate([currentCons, currentWealth, portfolioLayer_outputs, input_matrix[:,iTime+1, 0:self.numAssets]])  #if iTime<=input_matrix.shape[1]-2

          #outputWealthOut = Lambda(lambda x: self.updateWealthCons(x))(updateWealth_input)
          outputWealthOut = Lambda(lambda x: self.updateWealthCons(x),output_shape=(1,))(updateWealth_input)

          if iTime <= self.numPeriods_employ-1: #40

            prev_income = currentIncome
            #debug_tensors[f"prev_income"] = prev_income
            #currentIncome  = Lambda(lambda x: tf.math.multiply(tf.expand_dims(x[0][:, iTime + 1, -1], axis=-1), x[1]))([input_matrix, prev_income])
            currentIncome = Lambda(lambda x, t=iTime: tf.math.multiply(tf.expand_dims(x[0][:, t + 1, -1], axis=-1),x[1]),output_shape=(1,))([input_matrix, prev_income])
            currentWealth =  outputWealthOut + currentIncome

          else: currentWealth = outputWealthOut

          # Store tensors for debugging
          #debug_tensors[f"consWealthOut{iTime}"] = consWealthOut
          #debug_tensors[f"aftercurrentCons_{iTime}"] = currentCons
          #debug_tensors[f"outputWealthOut_{iTime}"] = outputWealthOut
          #debug_tensors[f"IncomeGrowth_{iTime}"] = Lambda(lambda x, t=iTime: tf.expand_dims(x[:, t + 1, -1], axis=-1),output_shape=(1,))(input_matrix)

         # Keras expects shapes without the batch dimensio
          #debug_tensors[f"beforeCurrentIncome_{iTime}"] = prev_income
          #debug_tensors[f"afterCurrentIncome_{iTime}"] = currentIncome
          #debug_tensors[f"aftercurrentWealth_{iTime}"] = currentWealth

        consWealthOut = Concatenate(axis=-1)([consWealthOut, currentWealth])   #等下记得改过来

        debug_tensors["final_consWealthOut"] = consWealthOut

        debug_outputs = list(debug_tensors.values())

        # Build the model
        intermediate_model = Model(inputs=input_matrix, outputs=debug_outputs)
        model = Model(inputs=input_matrix, outputs=consWealthOut)

        # Define optimizer with gradient clipping
        opt = keras.optimizers.Adam(self.parsSolution["learningRate"], clipnorm=1.0)

        # Compile the model
        model.compile(
            loss=lambda y_true, y_pred: self.neuralNetPolicyConsumLoss_2(y_true, y_pred, self.parsSolution["parUtil"], adjustFactor),
            optimizer=opt
        )

        # Define callbacks
        callbackList = [

            keras.callbacks.EarlyStopping(
                monitor='val_loss',
                min_delta=self.parsSolution["min_delta"],
                patience=self.parsSolution["parPatience"]
            ),

            keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=5,
                min_delta=self.parsSolution["min_delta"],
                min_lr=1e-6,
                verbose=1
            ),

          keras.callbacks.ModelCheckpoint(
                filepath='/content/drive/My Drive/ModelCheckpoints/Ex1/my_weights.weights.h5',
                monitor='val_loss',
                save_best_only=True,
                save_weights_only=True,
            ),
    ]

        """
        keras.callbacks.PickleCheckpoint(
              filepath='/content/drive/My Drive/ModelCheckpoints/Ex1/my_weights.pkl',  # where to save
              monitor='val_loss',
              verbose=1,
              save_best_only=True,  # only save if val_loss improves
              mode='min'
          ),
        """
        #PrintIntermediateValuesCallback_new(intermediate_model, dataTrain, self.numPeriods+1),
        #PrintInputMatrixCallback_new(dataTrain)
        #callbackList.append(GradientCheckCallback())

        # Fit the model
        history = model.fit(
            x=dataTrain,
            validation_data=dataValid,
            epochs=self.parsSolution["numEpochs"],
            batch_size=self.numSim,
            callbacks=callbackList
        )

        # Get the minimum validation loss
        min_val_loss = min(history.history['val_loss'])

        print("Model training is done")
        modelPars = model.get_weights()

        return modelPars, model

    def optPortfConsIncNeuralNetPolicy_normal_3(self):

        print("Running code for the solution using a policy function determined by a neural network: Consumption and portfolio decision with income")
        print("The setup is:")
        print("Model: {}".format(self.parsSolution['model']))
        print("Utility: {}".format(self.investor.utilFunc))

        # Define base directory for checkpoints
        base_dir = '/content/drive/My Drive/ModelCheckpoints/Ex1/'
        os.makedirs(base_dir, exist_ok=True)
        checkpoint_filepath = os.path.join(base_dir, "model.weights.h5")  # Fix filepath to end with .weights.h5

        # Ensure reproducible neural network fit
        if self.parsSolution.get("seedIndicator", 0) == 1:
            tf.random.set_seed(self.parsSolution["seed"])

        # Simulate the economy forward to get training and validation data
        if self.parsSolution.get("simDataPolicyPass") is None:
            simData = self.economy.simulation(self.parsSolution)
        else:
            simData = self.parsSolution["simDataPolicy"]

        numVars = simData.shape[1]
        numPeriodsSimTrain = self.parsSolution["numBatchesTrain"] * self.numSim * (self.numPeriods + 1)
        numPeriodsSimValidate = (self.parsSolution["numBatchesTrain"] + self.parsSolution["numBatchesVal"]) * self.numSim * (self.numPeriods + 1)

        dataTrain = keras.utils.timeseries_dataset_from_array(
        simData[:numPeriodsSimTrain, :],
        targets=simData[:numPeriodsSimTrain, 1],
        sequence_length=self.numPeriods + 1,
        sequence_stride=self.numPeriods + 1,
        batch_size=self.numSim  # Ensure no partial batches
        )

        dataValid = keras.utils.timeseries_dataset_from_array(
            simData[numPeriodsSimTrain:numPeriodsSimValidate, :],
            targets=simData[numPeriodsSimTrain:numPeriodsSimValidate, 1],
            sequence_length=self.numPeriods + 1,
            sequence_stride=self.numPeriods + 1,
            batch_size=self.numSim
        )

        # Debug dataset shape
        #for batch in dataTrain.take(1):
            #print("dataTrain batch shape:", batch[0].shape)  # Should be (500, self.numPeriods + 1, numVars)
        #for batch in dataValid.take(1):
            #print("dataValid batch shape:", batch[0].shape)

        # Define the model using Functional API
        input_matrix = Input(shape=(self.numPeriods + 1, numVars), dtype='float64', name='input_matrix')

        #input_wealth = Lambda(lambda x: tf.cast(self.parsSolution["startWealth"], tf.float64) * tf.ones((tf.shape(x)[0], 1)))(input_matrix)

        input_wealth = Lambda(lambda x: tf.cast(self.parsSolution["startWealth"], tf.float64) * tf.ones((tf.shape(x)[0], 1), dtype=tf.float64),
          output_shape=(1,),dtype='float64')(input_matrix)

        # Define layers
        hiddenLayer_1 = tf.keras.layers.Dense(
            units=self.parsSolution["numNodes"][0],
            activation=self.parsSolution["activation"][0],
            kernel_initializer=tf.keras.initializers.RandomNormal(
                mean=self.parsSolution["kernelInitializers"][0][0],
                stddev=self.parsSolution["kernelInitializers"][0][1]
            ),
            bias_initializer=tf.keras.initializers.Constant(
                value=self.parsSolution["biasInitializers"][0][0]
            )
        )

        hiddenLayer_2 = tf.keras.layers.Dense(
            units=self.parsSolution["numNodes"][1],
            activation=self.parsSolution["activation"][1],
            kernel_initializer=tf.keras.initializers.RandomNormal(
                mean=self.parsSolution["kernelInitializers"][1][0],
                stddev=self.parsSolution["kernelInitializers"][1][1]
            ),
            bias_initializer=tf.keras.initializers.Constant(
                value=self.parsSolution["biasInitializers"][1][0]
            )
        )

        portfolioLayer = tf.keras.layers.Dense(
            units=self.parsSolution["numNodes"][2],
            #activation=self.parsSolution["activation"][2],
            activation= self.custom_activation,
            kernel_initializer=tf.keras.initializers.RandomNormal(
                mean=self.parsSolution["kernelInitializers"][2][0],
                stddev=self.parsSolution["kernelInitializers"][2][1]
            ),
            bias_initializer=tf.keras.initializers.Constant(
                value=-self.parsSolution["biasInitializers"][2][0]
            )
        )

        consLayer = tf.keras.layers.Dense(
            units=self.parsSolution["numNodes"][3],
            activation=self.parsSolution["activation"][3],
            kernel_initializer=tf.keras.initializers.RandomNormal(
                mean=self.parsSolution["kernelInitializers"][3][0],
                stddev=self.parsSolution["kernelInitializers"][3][1]
            ),
            bias_initializer=tf.keras.initializers.Constant(
                value=-self.parsSolution["biasInitializers"][3][0]
            )
        )

        consWealthOut = Lambda(lambda x: tf.zeros((tf.shape(x)[0], tf.shape(x)[1]-1), dtype=tf.float64))(input_matrix)
        currentWealth = input_wealth

        #consWealthOut = tf.keras.backend.zeros((tf.keras.backend.shape(input_matrix)[0], input_matrix.shape[1]-1))

        #currentIncome = Lambda(lambda x: tf.cast(self.parsSolution["startIncome"], tf.float64) * tf.ones((tf.shape(x)[0], 1), dtype=tf.float64))(input_matrix)
        #currentWealth = Lambda(lambda inputs: inputs[0] + inputs[1])([currentWealth, currentIncome])
        #currentWealth = currentWealth

        debug_tensors = {}
        # Subjective discount factor adjustments
        #adjustFactor = (tf.pow(np.array(self.parsSolution["parUtil"][-1]), tf.cast(tf.range(0, self.numPeriods+1), dtype=tf.float32)))
        # Define adjustFactor as a TensorFlow constant (no need for Lambda)

        #adjustFactor = tf.pow(tf.constant(self.parsSolution["parUtil"][-1], dtype=tf.float64),tf.range(0, self.numPeriods+1, dtype=tf.float64))

        base  = 10.0 / self.numPeriods

        adjustFactor = tf.constant([base]*self.numPeriods + [1.0], dtype=tf.float64)

        # Recursive loop over time step
        for iTime in range(0, input_matrix.shape[1]-1):  #60

        #for iTime in range(0, input_matrix.shape[1]-1):
          #debug_tensors[f"beforecurrrentIncome_{iTime}"] = currentIncome
          #debug_tensors[f"beforecurrrentWealth_{iTime}"] = currentWealth

          hiddenLayer_input = tf.keras.layers.concatenate([
              input_matrix[:,iTime, self.parsSolution["statesStart"]:self.parsSolution["statesEnd"]], currentWealth/self.parsSolution["startWealth"]])

          hiddenLayer_outputs_1 = hiddenLayer_1(hiddenLayer_input)
          hiddenLayer_outputs_2 = hiddenLayer_2(hiddenLayer_outputs_1)

          # Layer 2: Portfolio weights
          portfolioLayer_outputs = portfolioLayer(hiddenLayer_outputs_2)

          # Layer 3: Consumption/wealth ratio
          consLayer_outputs = consLayer(hiddenLayer_outputs_2)
          #debug_tensors[f"consLayer_outputs_{iTime}"] = consLayer_outputs
          currentCons = currentWealth* consLayer_outputs

          #batch_size = Lambda(lambda x: tf.shape(x)[0])(consWealthOut) # Compute indices safely
          #indices = Lambda(lambda x: tf.expand_dims(tf.range(x), axis=-1))(batch_size)# Ensure `indices` and `iTime` are compatible
          #iTime_tensor = Lambda(lambda x: tf.ones_like(x) * iTime)(indices)# Concatenate indices properly
          #flattened_currentCons = Lambda(lambda x: tf.reshape(x, [-1]))(currentCons)
          # Compute indices dynamically for the current iTime

          def update_cons_wealth_out(args):
              cons_wealth_out, current_cons, i_time = args
              batch_size = tf.shape(cons_wealth_out)[0]
              batch_indices = tf.range(batch_size, dtype=tf.int64)
              i_time_tensor = tf.ones_like(batch_indices, dtype=tf.int64) * i_time
              indices = tf.stack([batch_indices, i_time_tensor], axis=-1)
              flattened_current_cons = tf.reshape(current_cons, [-1])
              return  tf.tensor_scatter_nd_update(cons_wealth_out, indices, flattened_current_cons)

          preWealthOut = consWealthOut
          consWealthOut  = Lambda(update_cons_wealth_out, output_shape=lambda input_shapes: input_shapes[0])(
              [preWealthOut, currentCons, tf.constant(iTime, dtype=tf.int64)]
          )

          if iTime == input_matrix.shape[1]-1:  #
            print("Condition met, breaking loop.")
            break

          #prev_consWealthOut = consWealthOut
          #consWealthOut = Lambda(lambda x: tf.tensor_scatter_nd_update(x[0], x[1], x[2]),output_shape=lambda input_shapes: input_shapes[0])([consWealthOut, indices, flattened_currentCons])

          updateWealth_input = tf.keras.layers.concatenate([currentCons*base, currentWealth, portfolioLayer_outputs, input_matrix[:,iTime+1, 0:self.numAssets]])  #if iTime<=input_matrix.shape[1]-2

          #outputWealthOut = Lambda(lambda x: self.updateWealthCons(x))(updateWealth_input)
          outputWealthOut = Lambda(lambda x: self.updateWealthCons(x),output_shape=(1,))(updateWealth_input)

          #if iTime <= self.numPeriods_employ-1: #40

            #prev_income = currentIncome
            #debug_tensors[f"prev_income"] = prev_income
            #currentIncome  = Lambda(lambda x: tf.math.multiply(tf.expand_dims(x[0][:, iTime + 1, -1], axis=-1), x[1]))([input_matrix, prev_income])
            #currentIncome = Lambda(lambda x, t=iTime: tf.math.multiply(tf.expand_dims(x[0][:, t + 1, -1], axis=-1),x[1]),output_shape=(1,))([input_matrix, prev_income])
            #currentWealth =  outputWealthOut

          #else: currentWealth = outputWealthOut
          currentWealth = outputWealthOut

          # Store tensors for debugging
          #debug_tensors[f"consWealthOut{iTime}"] = consWealthOut
          #debug_tensors[f"aftercurrentCons_{iTime}"] = currentCons
          #debug_tensors[f"outputWealthOut_{iTime}"] = outputWealthOut
          #debug_tensors[f"IncomeGrowth_{iTime}"] = Lambda(lambda x, t=iTime: tf.expand_dims(x[:, t + 1, -1], axis=-1),output_shape=(1,))(input_matrix)

         # Keras expects shapes without the batch dimensio
          #debug_tensors[f"beforeCurrentIncome_{iTime}"] = prev_income
          #debug_tensors[f"afterCurrentIncome_{iTime}"] = currentIncome
          #debug_tensors[f"aftercurrentWealth_{iTime}"] = currentWealth

        consWealthOut = Concatenate(axis=-1)([consWealthOut, currentWealth])   #等下记得改过来

        debug_tensors["final_consWealthOut"] = consWealthOut

        debug_outputs = list(debug_tensors.values())

        # Build the model
        intermediate_model = Model(inputs=input_matrix, outputs=debug_tensors)
        model = Model(inputs=input_matrix, outputs=consWealthOut)

        # Define optimizer with gradient clipping
        opt = keras.optimizers.Adam(self.parsSolution["learningRate"], clipnorm=1.0)

        # Compile the model
        model.compile(
            loss=lambda y_true, y_pred: self.neuralNetPolicyConsumLoss_2(y_true, y_pred, self.parsSolution["parUtil"], adjustFactor),
            optimizer=opt
        )

        # Define callbacks
        callbackList = [

            keras.callbacks.EarlyStopping(
                monitor='val_loss',
                min_delta=self.parsSolution["min_delta"],
                patience=self.parsSolution["parPatience"]
            ),

            keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=5,
                min_delta=self.parsSolution["min_delta"],
                min_lr=1e-6,
                verbose=1
            ),

          keras.callbacks.ModelCheckpoint(
                filepath='/content/drive/My Drive/ModelCheckpoints/Ex1/my_weights.weights.h5',
                monitor='val_loss',
                save_best_only=True,
                save_weights_only=True,
            ),
    ]

        """
        keras.callbacks.PickleCheckpoint(
              filepath='/content/drive/My Drive/ModelCheckpoints/Ex1/my_weights.pkl',  # where to save
              monitor='val_loss',
              verbose=1,
              save_best_only=True,  # only save if val_loss improves
              mode='min'
          ),
        """
        #PrintIntermediateValuesCallback_new(intermediate_model, dataTrain, self.numPeriods+1),
        #PrintInputMatrixCallback_new(dataTrain)
        #callbackList.append(GradientCheckCallback())

        # Fit the model
        history = model.fit(
            x=dataTrain,
            validation_data=dataValid,
            epochs=self.parsSolution["numEpochs"],
            batch_size=self.numSim,
            callbacks=callbackList
        )

        # Get the minimum validation loss
        min_val_loss = min(history.history['val_loss'])

        print("Model training is done")
        modelPars = model.get_weights()

        return modelPars, model


    def optPortfConsIncNeuralNetPolicy_normal_Parrel(self,process_id,result):

        print("Running code for the solution using a policy function determined by a neural network: Consumption and portfolio decision with income")
        print("The setup is:")
        print("Model: {}".format(self.parsSolution['model']))
        print("Utility: {}".format(self.investor.utilFunc))
        print(f"Process {process_id} (PID {os.getpid()}) started.")


        # Define base directory for checkpoints
        #base_dir = '/content/drive/My Drive/ModelCheckpoints/Ex1/'
        #os.makedirs(base_dir, exist_ok=True)
        #checkpoint_filepath = os.path.join(base_dir, "model.weights.h5")  # Fix filepath to end with .weights.h5

        # Ensure reproducible neural network fit
        if self.parsSolution.get("seedIndicator", 0) == 1:
            tf.random.set_seed(self.parsSolution["seed"])

        # Simulate the economy forward to get training and validation data
        if self.parsSolution.get("simDataPolicyPass") is None:
            simData = self.economy.simulation(self.parsSolution)
        else:
            simData = self.parsSolution["simDataPolicy"]

        numVars = simData.shape[1]
        numPeriodsSimTrain = self.parsSolution["numBatchesTrain"] * self.numSim * (self.numPeriods + 1)
        numPeriodsSimValidate = (self.parsSolution["numBatchesTrain"] + self.parsSolution["numBatchesVal"]) * self.numSim * (self.numPeriods + 1)

        dataTrain = keras.utils.timeseries_dataset_from_array(
        simData[:numPeriodsSimTrain, :],
        targets=simData[:numPeriodsSimTrain, 1],
        sequence_length=self.numPeriods + 1,
        sequence_stride=self.numPeriods + 1,
        batch_size=self.numSim  # Ensure no partial batches
        )

        dataValid = keras.utils.timeseries_dataset_from_array(
            simData[numPeriodsSimTrain:numPeriodsSimValidate, :],
            targets=simData[numPeriodsSimTrain:numPeriodsSimValidate, 1],
            sequence_length=self.numPeriods + 1,
            sequence_stride=self.numPeriods + 1,
            batch_size=self.numSim
        )

        # Debug dataset shape
        #for batch in dataTrain.take(1):
            #print("dataTrain batch shape:", batch[0].shape)  # Should be (500, self.numPeriods + 1, numVars)
        #for batch in dataValid.take(1):
            #print("dataValid batch shape:", batch[0].shape)

        # Define the model using Functional API
        input_matrix = Input(shape=(self.numPeriods + 1, numVars), dtype='float64', name='input_matrix')

        #input_wealth = Lambda(lambda x: tf.cast(self.parsSolution["startWealth"], tf.float64) * tf.ones((tf.shape(x)[0], 1)))(input_matrix)

        input_wealth = Lambda(lambda x: tf.cast(self.parsSolution["startWealth"], tf.float64) * tf.ones((tf.shape(x)[0], 1), dtype=tf.float64),
          output_shape=(1,),dtype='float64')(input_matrix)

        # Define layers
        hiddenLayer_1 = tf.keras.layers.Dense(
            units=self.parsSolution["numNodes"][0],
            activation=self.parsSolution["activation"][0],
            kernel_initializer=tf.keras.initializers.RandomNormal(
                mean=self.parsSolution["kernelInitializers"][0][0],
                stddev=self.parsSolution["kernelInitializers"][0][1]
            ),
            bias_initializer=tf.keras.initializers.Constant(
                value=self.parsSolution["biasInitializers"][0][0]
            )
        )

        hiddenLayer_2 = tf.keras.layers.Dense(
            units=self.parsSolution["numNodes"][1],
            activation=self.parsSolution["activation"][1],
            kernel_initializer=tf.keras.initializers.RandomNormal(
                mean=self.parsSolution["kernelInitializers"][1][0],
                stddev=self.parsSolution["kernelInitializers"][1][1]
            ),
            bias_initializer=tf.keras.initializers.Constant(
                value=self.parsSolution["biasInitializers"][1][0]
            )
        )

        portfolioLayer = tf.keras.layers.Dense(
            units=self.parsSolution["numNodes"][2],
            #activation=self.parsSolution["activation"][2],
            activation= self.custom_activation,
            kernel_initializer=tf.keras.initializers.RandomNormal(
                mean=self.parsSolution["kernelInitializers"][2][0],
                stddev=self.parsSolution["kernelInitializers"][2][1]
            ),
            bias_initializer=tf.keras.initializers.Constant(
                value=-self.parsSolution["biasInitializers"][2][0]
            )
        )

        consLayer = tf.keras.layers.Dense(
            units=self.parsSolution["numNodes"][3],
            activation=self.parsSolution["activation"][3],
            kernel_initializer=tf.keras.initializers.RandomNormal(
                mean=self.parsSolution["kernelInitializers"][3][0],
                stddev=self.parsSolution["kernelInitializers"][3][1]
            ),
            bias_initializer=tf.keras.initializers.Constant(
                value=-self.parsSolution["biasInitializers"][3][0]
            )
        )

        consWealthOut = Lambda(lambda x: tf.zeros((tf.shape(x)[0], tf.shape(x)[1]-1), dtype=tf.float64))(input_matrix)
        currentWealth = input_wealth

        #consWealthOut = tf.keras.backend.zeros((tf.keras.backend.shape(input_matrix)[0], input_matrix.shape[1]-1))

        currentIncome = Lambda(lambda x: tf.cast(self.parsSolution["startIncome"], tf.float64) * tf.ones((tf.shape(x)[0], 1), dtype=tf.float64))(input_matrix)
        #currentWealth = Lambda(lambda inputs: inputs[0] + inputs[1])([currentWealth, currentIncome])
        currentWealth = currentWealth + currentIncome

        debug_tensors = {}
        # Subjective discount factor adjustments
        #adjustFactor = (tf.pow(np.array(self.parsSolution["parUtil"][-1]), tf.cast(tf.range(0, self.numPeriods+1), dtype=tf.float32)))
        # Define adjustFactor as a TensorFlow constant (no need for Lambda)

        adjustFactor = tf.pow(tf.constant(self.parsSolution["parUtil"][-1], dtype=tf.float64),tf.range(0, self.numPeriods+1, dtype=tf.float64))

        # Recursive loop over time step
        for iTime in range(0, input_matrix.shape[1]-1):  #60

        #for iTime in range(0, input_matrix.shape[1]-1):
          #debug_tensors[f"beforecurrrentIncome_{iTime}"] = currentIncome
          #debug_tensors[f"beforecurrrentWealth_{iTime}"] = currentWealth

          hiddenLayer_input = tf.keras.layers.concatenate([
              input_matrix[:,iTime, self.parsSolution["statesStart"]:self.parsSolution["statesEnd"]], currentWealth/(self.parsSolution["startWealth"]+ self.parsSolution["startIncome"])])

          hiddenLayer_outputs_1 = hiddenLayer_1(hiddenLayer_input)
          hiddenLayer_outputs_2 = hiddenLayer_2(hiddenLayer_outputs_1)

          # Layer 2: Portfolio weights
          portfolioLayer_outputs = portfolioLayer(hiddenLayer_outputs_2)

          # Layer 3: Consumption/wealth ratio
          consLayer_outputs = consLayer(hiddenLayer_outputs_2)
          #debug_tensors[f"consLayer_outputs_{iTime}"] = consLayer_outputs
          currentCons = currentWealth*consLayer_outputs

          #batch_size = Lambda(lambda x: tf.shape(x)[0])(consWealthOut) # Compute indices safely
          #indices = Lambda(lambda x: tf.expand_dims(tf.range(x), axis=-1))(batch_size)# Ensure `indices` and `iTime` are compatible
          #iTime_tensor = Lambda(lambda x: tf.ones_like(x) * iTime)(indices)# Concatenate indices properly
          #indices = Lambda(lambda x: tf.keras.layers.Concatenate(axis=-1)([x[0], x[1]]))([indices, iTime_tensor])
          #flattened_currentCons = Lambda(lambda x: tf.reshape(x, [-1]))(currentCons)
          # Compute indices dynamically for the current iTime

          def update_cons_wealth_out(args):
              cons_wealth_out, current_cons, i_time = args
              batch_size = tf.shape(cons_wealth_out)[0]
              batch_indices = tf.range(batch_size, dtype=tf.int64)
              i_time_tensor = tf.ones_like(batch_indices, dtype=tf.int64) * i_time
              indices = tf.stack([batch_indices, i_time_tensor], axis=-1)
              flattened_current_cons = tf.reshape(current_cons, [-1])
              return  tf.tensor_scatter_nd_update(cons_wealth_out, indices, flattened_current_cons)

          preWealthOut = consWealthOut
          consWealthOut  = Lambda(update_cons_wealth_out, output_shape=lambda input_shapes: input_shapes[0])(
              [preWealthOut, currentCons, tf.constant(iTime, dtype=tf.int64)]
          )

          if iTime == input_matrix.shape[1]-1:  #
            print("Condition met, breaking loop.")
            break

          #prev_consWealthOut = consWealthOut
          #consWealthOut = Lambda(lambda x: tf.tensor_scatter_nd_update(x[0], x[1], x[2]),output_shape=lambda input_shapes: input_shapes[0])([consWealthOut, indices, flattened_currentCons])

          updateWealth_input = tf.keras.layers.concatenate([currentCons, currentWealth, portfolioLayer_outputs, input_matrix[:,iTime+1, 0:self.numAssets]])  #if iTime<=input_matrix.shape[1]-2

          #outputWealthOut = Lambda(lambda x: self.updateWealthCons(x))(updateWealth_input)
          outputWealthOut = Lambda(lambda x: self.updateWealthCons(x),output_shape=(1,))(updateWealth_input)

          if iTime <= self.numPeriods_employ-1: #40

            prev_income = currentIncome
            #debug_tensors[f"prev_income"] = prev_income
            #currentIncome  = Lambda(lambda x: tf.math.multiply(tf.expand_dims(x[0][:, iTime + 1, -1], axis=-1), x[1]))([input_matrix, prev_income])
            currentIncome = Lambda(lambda x, t=iTime: tf.math.multiply(tf.expand_dims(x[0][:, t + 1, -1], axis=-1),x[1]),output_shape=(1,))([input_matrix, prev_income])
            currentWealth =  outputWealthOut + currentIncome

          else: currentWealth = outputWealthOut

          # Store tensors for debugging
          #debug_tensors[f"consWealthOut{iTime}"] = consWealthOut
          #debug_tensors[f"aftercurrentCons_{iTime}"] = currentCons
          #debug_tensors[f"outputWealthOut_{iTime}"] = outputWealthOut
          #debug_tensors[f"IncomeGrowth_{iTime}"] = Lambda(lambda x, t=iTime: tf.expand_dims(x[:, t + 1, -1], axis=-1),output_shape=(1,))(input_matrix)

         # Keras expects shapes without the batch dimensio
          #debug_tensors[f"beforeCurrentIncome_{iTime}"] = prev_income
          #debug_tensors[f"afterCurrentIncome_{iTime}"] = currentIncome
          #debug_tensors[f"aftercurrentWealth_{iTime}"] = currentWealth

        consWealthOut = Concatenate(axis=-1)([consWealthOut, currentWealth])   #等下记得改过来

        debug_tensors["final_consWealthOut"] = consWealthOut

        debug_outputs = list(debug_tensors.values())

        # Build the model
        intermediate_model = Model(inputs=input_matrix, outputs=debug_outputs)
        model = Model(inputs=input_matrix, outputs=consWealthOut)

        # Define optimizer with gradient clipping
        opt = keras.optimizers.Adam(self.parsSolution["learningRate"], clipnorm=1.0)

        # Compile the model
        model.compile(
            loss=lambda y_true, y_pred: self.neuralNetPolicyConsumLoss_2(y_true, y_pred, self.parsSolution["parUtil"], adjustFactor),
            optimizer=opt
        )

        # Define callbacks
        callbackList = [
            keras.callbacks.EarlyStopping(
                monitor='val_loss',
                min_delta=self.parsSolution["min_delta"],
                patience=self.parsSolution["parPatience"]
            ),

            keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=5,
                min_delta=self.parsSolution["min_delta"],
                min_lr=1e-6,
                verbose=1
            ),

          keras.callbacks.ModelCheckpoint(
                self.get_checkpoint_filepath(process_id),  # where to save
                monitor='val_loss',
                verbose=1,
                save_best_only=True,  # only save if val_loss improves
                mode='min',
                save_weights_only=True
            ),

          #PrintIntermediateValuesCallback_new(intermediate_model, dataTrain, self.numPeriods+1),
          #PrintInputMatrixCallback_new(dataTrain)
    ]
        #callbackList.append(GradientCheckCallback())

        # Fit the model
        history = model.fit(
            x=dataTrain,
            validation_data=dataValid,
            epochs=self.parsSolution["numEpochs"],
            batch_size=self.numSim,
            callbacks=callbackList
        )

        # Get the minimum validation loss
        min_val_loss = min(history.history['val_loss'])

        print("Model training is done")
        modelPars = model.get_weights()
        weights_filepath = self.get_weights_filepath(process_id)
        with open(weights_filepath, 'wb') as f:
          pickle.dump(modelPars, f)

        results[process_id] = {
            'min_val_loss': min_val_loss,
            'weights': model.get_weights()  # Save model weights
        }

        print(f"Process {process_id}: Min validation loss = {min_val_loss}")


    def neuralNetPolicyConsumLoss_2(self, y_true, y_pred, parsUtil, adjustFactor):

        # Clip predicted values to prevent extreme values
        #y_pred_clipped = tf.clip_by_value(y_pred, 1e-6, 1e10) # Adjust clipping bounds as needed
        #tf.print("y_pred_clipped:", y_pred_clipped)
        #tf.print("consumption", y_pred[:5])
        #utility = self.investor.utility(y_pred+1e-4, parsUtil)  # Remove +0.001 if not needed

        utility = self.investor.utility(y_pred, parsUtil)

        #tf.print("Utility shape:", tf.shape(utility))
        #tf.print("Utility sample:", u)
        batch_loss = -adjustFactor * tf.reduce_mean(utility, axis=0)

        '''
        # Check for NaN or Inf in the loss
        if tf.reduce_any(tf.math.is_nan(batch_loss)) or tf.reduce_any(tf.math.is_inf(batch_loss)):
            tf.print("NaN or Inf detected in batch_loss. Returning a large value.")
            return tf.constant(1e10, dtype=batch_loss.dtype) # Return a large constant loss
        '''
       # Check for NaN or Inf in the loss (TensorFlow-safe)
        has_nan_or_inf = tf.logical_or(
          tf.reduce_any(tf.math.is_nan(batch_loss)),
          tf.reduce_any(tf.math.is_inf(batch_loss))
          )

        '''
       # Use tf.cond for conditional logic inside the graph
        batch_loss = tf.cond(
            has_nan_or_inf,
          lambda: tf.cast(tf.fill(tf.shape(batch_loss), 1e10), batch_loss.dtype),
          lambda: batch_loss
           )

        # Optionally print a warning message
        tf.cond(has_nan_or_inf,
              lambda: tf.print("⚠️ NaN or Inf detected in batch_loss. Returning large value."),
              lambda: tf.no_op()
                 )
        '''
        #tf.print("batch_loss shape:", tf.shape(batch_loss))
        #tf.print("batch_loss sample:",batch_loss[:5])
        #tf.debugging.check_numerics(tensor, "name")
        return 1000* tf.reduce_sum(batch_loss)

    def relu(self, x):
        return np.maximum(0, x)

    def tanh(self, x):
        return np.tanh(x)

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def softmax(self,x):

        e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))  # Stability adjustment
        return e_x / np.sum(e_x, axis=-1, keepdims=True)

    def custom_activation(self, x):

        u = 1.3 * tf.math.tanh(x)
        sum_u = tf.reduce_sum(u, axis=-1, keepdims=True)
        delta = (1.0 - sum_u) / 3.0
        return u + delta

    def custom_activation_2(self, x):

        u = 1.3 * np.tanh(x)
        sum_u = np.sum(u)
        delta = (1.0 - sum_u) / 3.0
        return u + delta

In [ ]:
utilFunc = "Power"
consume = 1

startWealth = 10.0
utilPars = [5.0, 1]

numPeriods_employ = 50
numPeriods_retire = 0
numPeriods = numPeriods_employ + numPeriods_retire # number of periods until maturity

n_paths = 200

numBatchesTrain = 200
numBatchesVal = 100 # number of batches used in the validation dataset, numBatches is split up between training and validation
numBatchesTest = 20
numBatches = numBatchesTrain + numBatchesVal + numBatchesTest

#K_returns = 101   # MUST match what you use inside simulate_primal_market_array
numSimPol = n_paths

numAssets = 2
numStates = 0
min_delta = 1e-6

economyPol = economy(numAssets, numStates,numPeriods_employ, numPeriods_retire, numSimPol)
modelUsed = "simulate_market_ou_array"
modelUsedPol = modelUsed

parsPol = {"model": modelUsedPol, "testDataPolicyPass": None, "simDataPolicyPass": 1,
           "statesStart": None, "statesEnd": None, "numBatches": numBatches, "numBatchesTrain": numBatchesTrain,
           "numBatchesVal": numBatchesVal,"numBatchesTest": numBatchesTest ,"startIncome": 0,"industry": None,"min_delta": min_delta,"seed":1234,
            "mu":(0.06, 0.08),"sigma_diag":(0.20, 0.25)}

simEconomyPol = economyPol.simulation(parsPol)
investorPol = investorTf(utilFunc)

In [ ]:
print(simEconomyPol[:,0:4])

[[0.93936283 1.00516831 1.00080032 0.        ]
 [1.00773453 1.04617138 1.00093559 0.02      ]
 [0.94407192 1.05045526 1.00140373 0.04      ]
 ...
 [0.91017195 1.03460305 1.00096774 0.94      ]
 [1.09370751 0.96618624 1.00108252 0.96      ]
 [1.00961623 0.97880067 1.00115632 0.98      ]]


In [ ]:
import matplotlib.pyplot as plt
!pip install statsmodels
##import statsmodels.api as sm

utilFunc = "Power"
consume = 1

startWealth = 0
utilPars = [10, 0.96]
iConsume = 1   # number of periods until maturity
numPeriods_employ = 40
numPeriods_retire = 20
numPeriods = numPeriods_employ + numPeriods_retire # number of periods until maturity

#timeToMaturity = 5 # time to maturity in years

numSimPolInput = 200
numSimPol = numSimPolInput # number of sequences in each batch for the policy function method  # number of batches used in the training dataset, numBatches is split up between training and validation
numBatchesTrain = 250
numBatchesVal = 100 # number of batches used in the validation dataset, numBatches is split up between training and validation
numBatchesTest = 100
numBatches = numBatchesTrain + numBatchesVal + numBatchesTest  # each batch contains numSim * numPeriods periods, with each batch containing numSim sequences of length numPeriods

numAssets = 3
numStates = 1
min_delta = 1e-6
normalizationTtm = numPeriods  + 1

economyPol = economy(numAssets, numStates,numPeriods_employ, numPeriods_retire, numSimPol)
modelUsed = "bootstrap_data_Ic_9_employ"
modelUsedPol = modelUsed

parsPol = {"model": modelUsedPol, "testDataPolicyPass": None, "simDataPolicyPass": 1,
           "statesStart": None, "statesEnd": None, "numBatches": numBatches, "numBatchesTrain": numBatchesTrain,
           "numBatchesVal": numBatchesVal,"numBatchesTest": numBatchesTest,"startIncome": 10,"industry": None,"min_delta": min_delta}

array, simData_employ = economyPol.simulation(parsPol)

modelUsedPol = "bootstrap_data_Ic_9_retire"
parsPol["model"] = modelUsedPol
simData_retire = economyPol.simulation(parsPol)
nCols = simData_employ.shape[1]

#We will accumulate chunks for each batch in this list
chunks_per_batch = []

for i in range(numBatches*numSimPol):

    # 1. Slice out the chunk from simData_employ for batch i
    start_e = i * (numPeriods_employ + 1)
    end_e = (i + 1) * (numPeriods_employ + 1)
    block_employ = simData_employ[start_e:end_e, :]

    # 2. Slice out the chunk from simData_retire for batch i
    start_r = i * numPeriods_retire
    end_r = (i + 1) * numPeriods_retire
    block_retire = simData_retire[start_r:end_r, :]

    # 3. Concatenate these two chunks vertically
    block_combined = np.concatenate([block_employ, block_retire], axis=0)

    # 4. Store in the list
    chunks_per_batch.append(block_combined)

# Finally, concatenate all batches along the row dimension
simEconomyPol= np.concatenate(chunks_per_batch, axis=0)

simEconomyPol[:, -1] = simEconomyPol[:, -1] / normalizationTtm

num_industries = 9
industry_names = ['Construction','Finance','Manufacturing','Mining','retail','Services', 'Comm. Util.','Wholesale','Government']
investorPol = investorTf(utilFunc)
file_path = '/content/drive/My Drive/project data/my_array.npy'
np.save(file_path, simEconomyPol)

In [ ]:
if consume == 0:

    numLayers = 2  # no more than two (no consumption)/ four (with consumption) are implemented yet.
    # number of nodes in the hidden layer
    numNodes = [4, numAssets]
    # activation functions for the layers, first two are for the portfolio weights, the second for the portfolio output,
    activation = ["relu", "softmax"]

    kernelInitializers = [[0.0, 0.2],[0.0,0.01]]
    biasInitializers = [[0.0],[0.5]],[[0.0], [0.0], [-2.0]]

else:

    parsPol["numAssets"] =2
    numAssets = parsPol["numAssets"]
    numLayers = 4  # no more than four (with consumption) are implemented yet.
    # number of nodes in the hidden layer
    numNodes = [10, 10, numAssets, 1]
    # activation functions for the layers, first two are for the portfolio weights, the second for the portfolio output,
    # set the second activation to "softmax" to restrict portfolio weights
    #activation = ["tanh", "relu","softmax", "sigmoid"]
    activation = ["tanh", "relu","softmax", "sigmoid"]  #allowing the short-sell case

    #kernelInitializers = [[0.0, 0.2],[0.0, 0.2],[0.0,0.2],[0.0,0.2]]
    kernelInitializers = [[0.0, 0.01],[0.0,0.01],[0.0,0.02],[0,0.01]]
    #kernelInitializers = [[0.0, 0.2],[0.0,0.2],[0.0,0.5],[0,0.5]]
    biasInitializers = [[0.01],[0.01],[0.001],[-0.01]]
    #biasInitializers = [[0.1], [0.1], [-1],[-0.1]]


numEpochs = 1000
parPatience = 8
learningRate = 0.01 #0.01
decaySteps = 200
decayRate = 1#0.8
startWealth = 10.0

parsOptPol = {"parUtil": utilPars, "startWealth": startWealth, "dataStates": None,"simDataPolicy": simEconomyPol,
              "objective": 'termWealth', "numAssets": numAssets, "numGrid": 1000,
              "numLayers": numLayers, "numNodes": numNodes, "numEpochs": numEpochs, "parPatience": parPatience,
              "learningRate": learningRate, "decaySteps": decaySteps, "decayRate": decayRate,
              "kernelInitializers": kernelInitializers, "biasInitializers": biasInitializers,
              "activation": activation, "optuna": '', "seed": 28, "seedIndicator": 1, "process_id": None}

# Include economy information
parsOptPol.update(parsPol)

weightOutAll = []
consWealthOutAll = []

consWealthRatio = np.zeros((numPeriods+1,1))

columns = ["No. Experiment", "Model", "Utility Function", "Consumption", "Utility Parameters", "Start Wealth", "val_loss"]

weightsOut_all_industry = np.empty((0, 1))
consWealthOut_al_industry = np.empty((0, 1))

In [ ]:
from multiprocessing import Manager, Process

In [ ]:
import pickle
from google.colab import drive
#drive.mount('/content/drive')
#file_path =  "/content/drive/My Drive/parameters//Construction/model_process_3.h5"
tf.keras.backend.set_floatx('float64')


if __name__ == "__main__":

  for i in np.array([1]):

    #print("Training the model: {}".format(industry_names[i]))

    for r in np.array([5]):

      parsOptPol["parUtil"][0] = r
      parsOptPol["startIncome"]= 0

      print("risk aversion is : {}".format(r))

      weightsOut_all_periods = []
      consWealthOut_all_periods = np.empty((0, 1))

      #parsOptPol["simDataPolicy"] = np.concatenate((simEconomyPol[:,[i]],simEconomyPol[:,8:11],simEconomyPol[:,[13]],simEconomyPol[:,[23]],simEconomyPol[:,[26]],simEconomyPol[:,[-1]],simEconomyPol[:,[indus_index]]),axis=1) #the past return not included case for the manufactury
      #parsOptPol["simDataPolicy"] = np.concatenate((simEconomyPol[:,[i]],simEconomyPol[:,[8+i]],simEconomyPol[:,[17]],simEconomyPol[:,[19+i]],simEconomyPol[:,[37+i]],simEconomyPol[:,[46+i]],simEconomyPol[:,[-1]],simEconomyPol[:,[27+i]]),axis=1)
      #parsOptPol["simDataPolicy"] = np.concatenate((simEconomyPol[:,[i]],simEconomyPol[:,[8+i]],simEconomyPol[:,[17]],simEconomyPol[:,[19+i]],simEconomyPol[:,[37+i]],simEconomyPol[:,[-1]],simEconomyPol[:,[27+i]]),axis=1)
      #parsOptPol["simDataPolicy"] = np.concatenate((simEconomyPol[:,[i]],simEconomyPol[:,[8+i]],simEconomyPol[:,[17]],simEconomyPol[:,[19+i]],simEconomyPol[:,[-1]],simEconomyPol[:,[27+i]]),axis=1)
      #numPeriodsSimTest_index = parsPol["numBatchesTest"] * numSimPol * (numPeriods + 1)

      #dataTest = simEconomyPol[-numPeriodsSimTest_index:, :]

      #dataTest = np.concatenate((dataTest[:,[i]],dataTest[:,[8+i]],dataTest[:,[17]],dataTest[:,[19+i]],dataTest[:,[-1]],dataTest[:,[22+i]]),axis=1)
      #dataTest = np.concatenate((dataTest[:,[i]],dataTest[:,[8+i]],dataTest[:,[17]],dataTest[:,[19+i]],dataTest[:,[-1]],dataTest[:,[27+i]]),axis=1)
      #dataTest = np.concatenate((dataTest[:,[i]],dataTest[:,[8+i]],dataTest[:,[17]],dataTest[:,[19+i]],dataTest[:,[37+i]],dataTest[:,[-1]],dataTest[:,[27+i]]),axis=1)
      #dataTest = np.concatenate((dataTest[:,[i]],dataTest[:,[8+i]],dataTest[:,[17]],dataTest[:,[19+i]],dataTest[:,[37+i]],dataTest[:,[46+i]],dataTest[:,[-1]],dataTest[:,[27+i]]),axis=1)
      #dataTest = np.concatenate((dataTest[:,[i]],dataTest[:,8:11],dataTest[:,[12]],dataTest[:,[23]],dataTest[:,[25]],dataTest[:,[-1]],dataTest[:,[indus_index]]),axis=1)
      #dataTest = np.concatenate((dataTest[:,[i]],dataTest[:,8:11],dataTest[:,[13]],dataTest[:,[23]],dataTest[:,[26]], dataTest[:,[-1]],dataTest[:,[indus_index]]),axis=1)
      #dataTest = np.reshape(dataTest,(-1,61,numAssets+numStates+2))

      #parsOptPol["industry"] = industry_names[i]
      parsOptPol["simDataPolicy"] = simEconomyPol
      modelPol = portfolioDiscreteNew(investorPol, economyPol, parsOptPol)

      """
      with Manager() as manager:
        results = manager.dict()

        # Create processes
        processes = []
        for j in range(0,3):
            p = Process(target=modelPol.optPortfConsIncNeuralNetPolicy_normal_Parrel,args=(j,results))
            processes.append(p)
            p.start()

        # Wait for all processes to finish
        for p in processes:
            p.join()

      """
      par, model= modelPol.optPortfConsIncNeuralNetPolicy_normal_3()

      #with open(file_path, 'wb') as f:
        #pickle.dump(par, f)


risk aversion is : 5
Running code for the solution using a policy function determined by a neural network: Consumption and portfolio decision with income
The setup is:
Model: simulate_market_ou_array
Utility: Power
Epoch 1/1000
200/200 ━━━━━━━━━━━━━━━━━━━━ 61s 122ms/step - loss: 8121538.7083 - val_loss: 7657.5731 - learning_rate: 0.0100
Epoch 2/1000
200/200 ━━━━━━━━━━━━━━━━━━━━ 13s 63ms/step - loss: 7591.5846 - val_loss: 7116.6744 - learning_rate: 0.0100
Epoch 3/1000
200/200 ━━━━━━━━━━━━━━━━━━━━ 17s 83ms/step - loss: 6954.8322 - val_loss: 6732.6223 - learning_rate: 0.0100
Epoch 4/1000
200/200 ━━━━━━━━━━━━━━━━━━━━ 16s 58ms/step - loss: 6844.5031 - val_loss: 7164.9674 - learning_rate: 0.0100
Epoch 5/1000
200/200 ━━━━━━━━━━━━━━━━━━━━ 12s 59ms/step - loss: 6726.7813 - val_loss: 6870.4402 - learning_rate: 0.0100
Epoch 6/1000
200/200 ━━━━━━━━━━━━━━━━━━━━ 13s 64ms/step - loss: 6726.7038 - val_loss: 6528.6023 - learning_rate: 0.0100
Epoch 7/1000
200/200 ━━━━━━━━━━━━━━━━━━━━ 13s 63ms/step - los

In [ ]:
def merton_weights(returns, gamma=5, rolling='expanding'):
    """
    Compute Merton optimal portfolio weights for two risky assets + one risk-free asset.

    """
    returns = np.asarray(returns, dtype=float)
    risky   = returns[:, :2]          # A and B
    rf_vec  = returns[:, 2]           # risk-free

    if rolling == 'full':
        mu_hat    = risky.mean(axis=0)
        sigma_hat = np.cov(risky, rowvar=False)
        excess    = mu_hat - rf_vec.mean()
        w_risky   = np.linalg.inv(sigma_hat) @ excess / gamma
        w_rf      = 1.0 - w_risky.sum()
        return np.append(w_risky, w_rf)

In [ ]:
def batch_merton_mean(dataTest, gamma=5, rolling='full'):
    """
    Compute Merton optimal weights for every 2-D slice of a 3-D array
    and return their sample mean.

    Parameters
    ----------
    weights_3d : np.ndarray, shape (N, T, 3)
        A 3-D array where each slice weights_3d[i] is a (T×3) matrix of
        returns for two risky assets + one risk-free asset.
    gamma : float, default 5
        Relative risk-aversion parameter passed to merton_weights().
    rolling : {'full','expanding'}, default 'full'
        • 'full'      → one weight vector per slice (long-run estimate).
        • 'expanding' → a (T×3) matrix per slice; the function will take
                        the *last* row (i.e., the most recent weights)
                        before averaging across slices.

    Returns
    -------
    all_weights : np.ndarray
        Stack of the individual optimal‐weight vectors, shape (N, 3).
    mean_weight : np.ndarray
        The sample-mean optimal weight vector, shape (3,).
    """
    # Ensure we have a proper numpy array
    weights_3d = np.asarray(dataTest, dtype=float)

    all_weights = []
    for idata in dataTest:
        w = merton_weights(idata, gamma=gamma, rolling='full')
        # keep only the final allocation if rolling='expanding'
        w_vec = w[-1] if rolling == 'expanding' else w
        all_weights.append(w_vec)

    all_weights = np.vstack(all_weights)          # shape (N, 3)
    mean_weight = all_weights.mean(axis=0)        # shape (3,)

    return mean_weight

In [ ]:
batch_merton_mean(dataTest[:,:,0:3], gamma=5, rolling='full')

In [ ]:
import pickle
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
file_path = '/content/drive/My Drive/without_predictor/parameters/5/Finance/risk_income_2/model_process_52.pkl'
#model.load_weights(file_path)

with open(file_path, 'rb') as f:
    par = pickle.load(f)  # Load the saved parameter object

In [ ]:
consOut_list = []
weightsOut_all_periods = []
wealthOut_all_periods = np.empty((0, 1))
consWealthOut_all_periods = np.empty((0, 1))

iConsume = 1
numPeriods = 60
base = np.array(utilPars[-1])  # Ensure it's a NumPy array (or just use float)

# Create the exponent range (from 0 to numPeriods)
exponents = np.arange(0, numPeriods + 1, dtype=np.float32)  # Ensure float type 61 ones
# Compute the power
adjustFactor = np.power(base, exponents)  # Equivalent to tf.pow()

for idata in dataTest:

    (weightsOut, wealthOut, consOut, consWealthOut) = modelPol.policyFunctionConsIncEval(par, idata)
    #consWealthOut[-1,0] = 1.0
    consOut[-1,0] = wealthOut[-2,0]
    weightsOut_all_periods.append(weightsOut)
    consWealthOut_all_periods = np.append(consWealthOut_all_periods, consWealthOut, axis=0)

    wealthOut_all_periods = np.append(wealthOut_all_periods,wealthOut,axis=0)

    consOut_squeezed = consOut.squeeze()
    #consOut_extended = np.append(consOut_squeezed, wealthOut[-1])
    #consOut_list.append(consOut_extended)
    consOut_list.append(consOut_squeezed)

In [ ]:
weights_stack = np.stack(weightsOut_all_periods, axis=0)

# 3) take the pointwise mean across iterations
#    result has shape (n_period, n_asset)
weights_mean = weights_stack.mean(axis=0)

#print("stacked shape:", weights_stack.shape)
print("mean shape:   ", weights_mean)

In [ ]:
import pandas as pd
import altair as alt

# Load the data into a DataFrame
data = pd.DataFrame(weights_mean)

data.columns = ['service portfolio', 'Market portfolio_ex_service', 'risk_free asset']

# Reset index and melt the DataFrame for Altair
data['Period'] = range(1, len(data) + 1)
data_melted = data.melt(id_vars='Period', var_name='Asset', value_name='Weight')

# Define the academic style directly in the chart configuration
chart = alt.Chart(data_melted).mark_area().encode(
    x=alt.X('Period', axis=alt.Axis(title='Period', titleFont='serif', labelFont='serif', grid=False, tickSize=0, domain=True, domainWidth=0.8, labelPadding=8)),
    y=alt.Y('Weight', axis=alt.Axis(title='Weight', titleFont='serif', labelFont='serif', grid=False, tickSize=0, domain=True, domainWidth=0.8, labelPadding=8)),
    color=alt.Color('Asset', scale=alt.Scale(range=['#1f77b4', '#ff7f0e', '#2ca02c']), legend=alt.Legend(title='Asset', titleFont='serif', labelFont='serif', orient='top', padding=10, titlePadding=5, labelFontSize=10, titleFontSize=12)),
    tooltip=['Period', 'Asset', 'Weight']
).properties(
    title=alt.Title('Portfolio Composition Over Time', font='serif', fontSize=14, anchor='start', offset=10)
).configure(
    font='serif',
    view=alt.ViewConfig(strokeWidth=0) # Remove the default border around the plot
).configure_title(
    fontSize=14,
    font='serif',
    anchor='start',
    offset=10
).configure_axis(
    labelFont='serif',
    titleFont='serif',
    grid=False,
    tickSize=0,
    domain=True,
    domainWidth=0.8,
    labelPadding=8
).configure_legend(
    labelFont='serif',
    titleFont='serif',
    orient='top',
    padding=10,
    titlePadding=5,
    labelFontSize=10,
    titleFontSize=12
).configure_mark(
    tooltip=True,
    strokeWidth=0.8
).interactive()

# Enable the Colab renderer
alt.renderers.enable('colab')

# Show the chart
chart

In [ ]:
print("mean shape:   ", weights_stack[800,:,:])

In [ ]:
#import matplotlib.pyplot as plt
wealthOut_all_periods = np.reshape(wealthOut_all_periods,(-1, 62))
wealthOut_all_periods[:,-1] = 0

# Compute quantiles along the first axis (per column)
q05 = np.quantile(wealthOut_all_periods, 0.05, axis=0)
q50 = np.quantile(wealthOut_all_periods, 0.50, axis=0)  # median
q_mean = np.mean(wealthOut_all_periods,axis=0)
q95 = np.quantile(wealthOut_all_periods, 0.95, axis=0)

# Plot the quantiles
x = np.arange(wealthOut_all_periods.shape[1])
plt.plot(x, q05, label="5th Percentile", linestyle="--")
plt.plot(x, q50, label="Median", linestyle="-")
plt.plot(x, q_mean, label="Mean", linestyle="-")
plt.plot(x, q95, label="95th", linestyle="--")

plt.title("wealth for investor with gamma5")
plt.xlabel("time")
plt.ylabel("Wealth")
plt.legend()
plt.grid(True)
plt.ylim(0, 800)
plt.tight_layout()
plt.show()

In [ ]:
print(np.mean(wealthOut_all_periods, axis=0))

In [ ]:
consOut_array = np.array(consOut_list)
utility_values = investorPol.utility(consOut_array, utilPars)  # Assuming y_pred is already a NumPy array

# Compute the mean along axis 0
mean_utility = np.mean(utility_values, axis=0)
# Compute the batch loss
batch_loss = np.sum(adjustFactor * mean_utility)

weightsOut_mean = np.mean(weightsOut_all_periods, axis=0)
consWealthOut_all_periods = np.reshape(consWealthOut_all_periods, (-1, 61))
consWealthOutAll_mean = np.mean(consWealthOut_all_periods, axis=0)

print(batch_loss*1000)

In [ ]:
import seaborn as sns


# Prep (same as yours)
consWealthOut_all_periods = np.reshape(consWealthOut_all_periods, (-1, 61))
mean_vals = np.mean(consWealthOut_all_periods[:, :-1], axis=0)
std_vals  = np.std(consWealthOut_all_periods[:, :-1],  axis=0)
T = len(mean_vals)  # number of periods on the x-axis
x = np.arange(T)

# Styling: no grid, compact context, clean spines
sns.set_style("white")          # no background grid
sns.set_context("paper")        # smaller fonts/lines for compact figures

# Smaller figure
fig, ax = plt.subplots(figsize=(5, 3.5))

# Violin plot (no inner bars)
sns.violinplot(
    data=consWealthOut_all_periods[:, :-1],
    inner=None,
    color="skyblue",
    alpha=0.6,
    ax=ax,
    linewidth=0,        # <-- no edge stroke
    cut=0,
)

# Overlay mean ± std
ax.plot(x, mean_vals, label="Mean", linewidth=1.8)
ax.fill_between(x, mean_vals - std_vals, mean_vals + std_vals, alpha=0.2, label="±1 Std. Dev.")

# X ticks every 5
ticks = np.arange(0, T, 5)
ax.set_xticks(ticks)
ax.set_xticklabels(ticks)

# Labels/title/legend
#ax.set_title("Distribution of Consumption–Wealth Outputs Across Periods")
ax.set_xlabel("Period")
ax.set_ylabel("Consumption / Wealth Ratio")
ax.legend(frameon=False)

# No grid
ax.grid(False)

# Clean look
sns.despine(ax=ax)
fig.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def to_3d(weights_list, periods=60):
    items = []
    for w in weights_list:
        a = np.asarray(w, dtype=float)
        # squeeze then ensure column shape
        a = np.squeeze(a)
        if a.ndim == 1:
            a = a.reshape(-1, 1)
        elif a.ndim > 2:
            a = a.reshape(a.shape[0], -1)[:, :1]  # keep first column
        # pad/truncate to target length
        if a.shape[0] < periods:
            pad = np.full((periods - a.shape[0], a.shape[1]), np.nan)
            a = np.vstack([a, pad])
        elif a.shape[0] > periods:
            a = a[:periods, :]
        # ensure single column
        if a.shape[1] != 1:
            a = a[:, :1]
        items.append(a)
    return np.stack(items, axis=0)  # (N, periods, 1)

# --- build clean array ---
arr = to_3d(weightsOut_all_periods, periods=60)   # (N, 60, 1)
weights_2d = arr[:, :, 0]                         # (N, 60)

# --- stats (ignore NaNs) ---
means = np.nanmean(weights_2d, axis=0)
stds  = np.nanstd(weights_2d, axis=0, ddof=1)

# --- violin data per period (drop NaNs) ---
data_per_period = [weights_2d[:, j][~np.isnan(weights_2d[:, j])] for j in range(weights_2d.shape[1])]

# --- plot ---
fig = plt.subplots(figsize=(5, 3.5))

parts = plt.violinplot(
    data_per_period,
    positions=np.arange(len(data_per_period)),
    showmeans=False, showextrema=False, widths=0.8
)

# purple/blue aesthetics
for pc in parts['bodies']:
    pc.set_facecolor('#6A5ACD')   # SlateBlue
    pc.set_edgecolor('#483D8B')   # DarkSlateBlue
    pc.set_alpha(0.55)

plt.plot(np.arange(len(means)), means, color='#1E90FF', linewidth=2.2, label='Mean weight')
plt.fill_between(np.arange(len(means)), means - stds, means + stds,
                 color='#87CEFA', alpha=0.25, label='±1 std')

#plt.title("Distribution of Optimal Portfolio Weights Over Time")
plt.xlabel("Period")
plt.ylabel("Optimal Weight")
plt.ylim(0.0, 1.0)           # fixed y-axis as requested
plt.grid(alpha=0.25, linestyle='--')
plt.legend(frameon=False)
plt.tight_layout()
plt.show()


In [ ]:
consOut_array_1 = consOut_array[:,:-1]

print(np.mean(consOut_array,axis= 0))

In [ ]:
# Compute quantiles along the first axis (per column)
q05 = np.quantile(consOut_array, 0.05, axis=0)
q50 = np.quantile(consOut_array, 0.50, axis=0)  # median
q_mean = np.mean(consOut_array,axis=0)
q95 = np.quantile(consOut_array, 0.95, axis=0)

# Plot the quantiles
x = np.arange(consOut_array.shape[1])
plt.plot(x, q05, label="5th Percentile", linestyle="--")
plt.plot(x, q50, label="Median", linestyle="-")
plt.plot(x, q_mean, label="Mean", linestyle="--")
plt.plot(x, q95, label="95th Percentile", linestyle="--")

plt.title("Consumption for investor with gamma5")
plt.xlabel("time")
plt.ylabel("Consumption")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.ylim(0, 200)          # force y-axis from 0 to 200
# or, equivalently:
# ax = plt.gca()
# ax.set_ylim(0, 200)


plt.show()

In [ ]:
first_period_data = dataTest[6, :, :]

# Compute mean across all 200 samples (resulting in shape: (1,6))
#mean_first_period = np.mean(first_period_data, axis=0, keepdims=True)
#mean_all_array = np.mean(dataTest, axis=(0,1), keepdims=True)
#print(mean_first_period)
print(first_period_data)

In [ ]:
consOut_array = np.array(consOut_list)
utility_values = investorPol.utility(consOut_array, utilPars)  # Assuming y_pred is already a NumPy array

# Compute the mean along axis 0
mean_utility = np.mean(utility_values, axis=0)
# Compute the batch loss
batch_loss = np.sum(adjustFactor * mean_utility)

weightsOut_mean = np.mean(weightsOut_all_periods, axis=0)
consWealthOut_all_periods = np.reshape(consWealthOut_all_periods, (-1, 61))
consWealthOutAll_mean = np.mean(consWealthOut_all_periods, axis=0)

print(batch_loss*1000)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

first_data = dataTest[700, :, :]
# Step 1: Extract the original mean_first_period (shape: (1,7))
original_mfp = first_data  # Shape: (1,7)

# Step 2: Define the different values for the 6th element (index 5)
time = [30,40,60]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']  # Using more elegant academic colors
markers = ['o', 's', '^']  # Different markers for clarity
linestyles = ['-', '--', ':']  # Line styles for differentiation
labels = ['Period 30', 'Period 40', 'Period 60']

# Step 3: Generate values for the 5th element (index 4)
num_points = 50  # Number of points to evaluate
x_values = np.linspace(0.0,0.06, num_points)

# Step 4: Create figure with high DPI
plt.figure(figsize=(8,5), dpi=300)  # High resolution for publication

# Step 5: Loop over different values of the 6th element
for i, sixth_value in enumerate(time):

    # Call the model function
    y_values = modelPol.policyFunctionConsIncEval_snap2(par,original_mfp,sixth_value,4)

    # Plot the results for this setting
    plt.plot(x_values, y_values, marker=markers[i], linestyle=linestyles[i],
             color=colors[i], label=labels[i], markersize=6, linewidth=2)

# Step 6: Format the plot for an academic paper
plt.xlabel("d/p Fin", fontsize=14, fontweight='bold')
plt.ylabel("Finance Portfolio Weight", fontsize=14, fontweight='bold')
plt.title("Impact of Finance Industry d/p on Finance Portfolio Weight", fontsize=16, fontweight='bold')
plt.legend(fontsize=12, loc='upper left', frameon=True, edgecolor='black')  # Better legend positioning
plt.grid(True, linestyle='--', alpha=0.7)  # Dotted grid lines for readability

# Save figure as high-quality PDF
plt.savefig("academic_plot.pdf", format="pdf", bbox_inches="tight")

# Show the plot
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

first_data = dataTest[800, :, :]
# Step 1: Extract the original mean_first_period (shape: (1,7))
original_mfp = first_data  # Shape: (1,7)

# Step 2: Define the different values for the 6th element (index 5)
time = [15,40,60]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']  # Using more elegant academic colors
markers = ['o', 's', '^']  # Different markers for clarity
linestyles = ['-', '--', ':']  # Line styles for differentiation
labels = ['Period 30', 'Period 40', 'Period 60']

# Step 3: Generate values for the 5th element (index 4)
num_points = 50  # Number of points to evaluate
x_values = np.linspace(0.0,0.06, num_points)

# Step 4: Create figure with high DPI
plt.figure(figsize=(8,5), dpi=300)  # High resolution for publication

# Step 5: Loop over different values of the 6th element
for i, sixth_value in enumerate(time):

    # Call the model function
    y_values = modelPol.policyFunctionConsIncEval_snap2(par,original_mfp,sixth_value,3)

    # Plot the results for this setting
    plt.plot(x_values, y_values, marker=markers[i], linestyle=linestyles[i],
             color=colors[i], label=labels[i], markersize=6, linewidth=2)

# Step 6: Format the plot for an academic paper
plt.xlabel("d/p Fin", fontsize=14, fontweight='bold')
plt.ylabel("Finance Portfolio Weight", fontsize=14, fontweight='bold')
plt.title("Impact of market d/p on market Portfolio Weight", fontsize=16, fontweight='bold')
plt.legend(fontsize=12, loc='upper left', frameon=True, edgecolor='black')  # Better legend positioning
plt.grid(True, linestyle='--', alpha=0.7)  # Dotted grid lines for readability

# Save figure as high-quality PDF
plt.savefig("academic_plot.pdf", format="pdf", bbox_inches="tight")

# Show the plot
plt.show()

In [ ]:
x_grid = np.linspace(0.0, 0.05, 41)
z_grid = np.linspace(1.0, 40.0, 41)
times = [15,40,60]

modelPol.plot_policy_surfaces_3in1(par, times, modelPol.eval_equity_weight_from_nn, dataTest, z_grid, x_grid,
                                      fname="policy_surfaces_3in1.png")

In [ ]:
import pandas as pd
import altair as alt

# Load the data into a DataFrame
data = pd.DataFrame(np.array([

])
)

data.columns = ['Construction portfolio', 'Market portfolio_ex_construct', 'risk_free asset']

# Reset index and melt the DataFrame for Altair
data['Period'] = range(1, len(data) + 1)
data_melted = data.melt(id_vars='Period', var_name='Asset', value_name='Weight')

# Define the academic style directly in the chart configuration
chart = alt.Chart(data_melted).mark_area().encode(
    x=alt.X('Period', axis=alt.Axis(title='Period', titleFont='serif', labelFont='serif', grid=False, tickSize=0, domain=True, domainWidth=0.8, labelPadding=8)),
    y=alt.Y('Weight', axis=alt.Axis(title='Weight', titleFont='serif', labelFont='serif', grid=False, tickSize=0, domain=True, domainWidth=0.8, labelPadding=8)),
    color=alt.Color('Asset', scale=alt.Scale(range=['#1f77b4', '#ff7f0e', '#2ca02c']), legend=alt.Legend(title='Asset', titleFont='serif', labelFont='serif', orient='top', padding=10, titlePadding=5, labelFontSize=10, titleFontSize=12)),
    tooltip=['Period', 'Asset', 'Weight']
).properties(
    title=alt.Title('Portfolio Composition Over Time', font='serif', fontSize=14, anchor='start', offset=10)
).configure(
    font='serif',
    view=alt.ViewConfig(strokeWidth=0) # Remove the default border around the plot
).configure_title(
    fontSize=14,
    font='serif',
    anchor='start',
    offset=10
).configure_axis(
    labelFont='serif',
    titleFont='serif',
    grid=False,
    tickSize=0,
    domain=True,
    domainWidth=0.8,
    labelPadding=8
).configure_legend(
    labelFont='serif',
    titleFont='serif',
    orient='top',
    padding=10,
    titlePadding=5,
    labelFontSize=10,
    titleFontSize=12
).configure_mark(
    tooltip=True,
    strokeWidth=0.8
).interactive()

# Enable the Colab renderer
alt.renderers.enable('colab')

# Show the chart
chart

In [ ]:
import altair as alt
data = pd.DataFrame(np.array([
   ]))
# Column names
data.columns = ['Construction portfolio', 'Market portfolio', 'risk_free asset']

# Long format for Altair
data['Period'] = range(1, len(data) + 1)
data_melted = data.melt(id_vars='Period', var_name='Asset', value_name='Weight')

# --- Put Market portfolio at the bottom ---
order_map = {'Market portfolio': 0, 'Finance portfolio': 1, 'risk_free asset': 2}
data_melted['stack_order'] = data_melted['Asset'].map(order_map)

chart = alt.Chart(data_melted).mark_area().encode(
    x=alt.X('Period:Q', axis=alt.Axis(title='Period', titleFont='serif', labelFont='serif',
                                      grid=False, tickSize=0, domain=True, domainWidth=0.8, labelPadding=8)),
    y=alt.Y('Weight:Q', stack='zero',
            axis=alt.Axis(title='Weight', titleFont='serif', labelFont='serif',
                          grid=False, tickSize=0, domain=True, domainWidth=0.8, labelPadding=8)),
    color=alt.Color(
        'Asset:N',
        # keep legend colors fixed and in the same order as stacking
        scale=alt.Scale(
            domain=list(order_map.keys()),
            range=['#ff7f0e', '#1f77b4', '#2ca02c']  # Market (bottom) = orange, Finance = blue, RF = green
        ),
        legend=alt.Legend(title='Asset', titleFont='serif', labelFont='serif',
                          orient='top', padding=10, titlePadding=5, labelFontSize=10, titleFontSize=12)
    ),
    order=alt.Order('stack_order:Q')  # <-- controls bottom→top stacking
).properties(
    title=alt.Title('Portfolio Composition Over Time', font='serif', fontSize=14, anchor='start', offset=10)
).configure(
    font='serif',
    view=alt.ViewConfig(strokeWidth=0)
).configure_title(
    fontSize=14, font='serif', anchor='start', offset=10
).configure_axis(
    labelFont='serif', titleFont='serif', grid=False, tickSize=0, domain=True, domainWidth=0.8, labelPadding=8
).configure_legend(
    labelFont='serif', titleFont='serif', orient='top', padding=10, titlePadding=5, labelFontSize=10, titleFontSize=12
).configure_mark(
    tooltip=True, strokeWidth=0.8
).interactive()

alt.renderers.enable('colab')
chart


In [ ]:
def calculate_certainty_equivalent(indirect_utility, gamma, periods, discount_factor):

    # Compute the discount factor sum
    discount_factor_sum = sum(discount_factor ** t for t in range(periods))

    # Calculate CE for CRRA utility (gamma != 1)
    if gamma != 1:
        ce = ((1 - gamma) * indirect_utility / discount_factor_sum) ** (1 / (1 - gamma))
    else:
        # Calculate CE for logarithmic utility (gamma == 1)
        ce = (indirect_utility / discount_factor_sum) ** (1 / periods)

    return ce


# Example Inputs
indirect_utility =  -0.147623/1000 # Aggregated utility (example value)
gamma = 5           # Risk aversion coefficient
periods = 61                 # Number of periods
discount_factor = 0.96       # Discount factor (beta)

# Calculate Certainty Equivalent
certainty_equivalent = calculate_certainty_equivalent(
    indirect_utility, gamma, periods, discount_factor
)

print(f"Certainty Equivalent (CE): {certainty_equivalent:.2f}")

In [ ]:
Certainty Equivalent